In [ ]:
import os

# Search the current directory and all subfolders for any .zip files
for root, dirs, files in os.walk("."):
    for f in files:
        if f.lower().endswith(".zip"):
            print(f"📦 Found ZIP file: {os.path.join(root, f)}")





In [ ]:
import os

# Show current working directory
print("Current Directory:", os.getcwd())

# List all files
print("Files:", os.listdir())

In [ ]:
# ✅ Step 1 & 2: Extract ZIP and Clean Labels

import zipfile  
import os  
import pandas as pd  
import re

# ZIP file is in same folder as notebook
zip_path = "trainzip.zip"
extract_dir = "train_data"

# Extract to "train_data" folder
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("✅ Extracted to:", extract_dir)

# Get all image paths inside extracted directory (recursively)
image_paths = []
for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            image_paths.append(os.path.join(root, file))

print(f"🖼 Found {len(image_paths)} images")

# Extract label from filename
def extract_label(filename):
    base = os.path.basename(filename)
    return re.match(r'[a-zA-Z]+', base).group(0)

# Create DataFrame
df = pd.DataFrame({
    "filepath": image_paths,
    "label": [extract_label(p) for p in image_paths]
})

# Encode labels
label_to_index = {label: idx for idx, label in enumerate(sorted(df["label"].unique()))}
df["label_encoded"] = df["label"].map(label_to_index)

# Show results
print("\n✅ Label Mapping:", label_to_index)
print("\n🧾 Sample data:")
print(df.head())

from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
import torch

class WasteDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.loc[idx, "filepath"]
        label = self.data.loc[idx, "label_encoded"]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label)
        


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Split your dataframe
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label_encoded"], random_state=42)

# Define image transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),     # Resize to MobileNetV2 input size
    transforms.ToTensor(),             # Convert to tensor
    transforms.Normalize(              # Normalize using ImageNet stats
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Create datasets
train_dataset = WasteDataset(train_df, transform=transform)
val_dataset = WasteDataset(val_df, transform=transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

print(f"✅ Train images: {len(train_dataset)}")
print(f"✅ Validation images: {len(val_dataset)}")



In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("📟 Using device:", device)

# Load pre-trained MobileNetV2
model = models.mobilenet_v2(pretrained=True)

# Replace the classifier (output layer) to match our 4 classes
model.classifier[1] = nn.Linear(model.last_channel, 4)  # 4 = num_classes
model = model.to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)


In [ ]:
import time

def train_model(model, criterion, optimizer, train_loader, val_loader, device, num_epochs=5):
    for epoch in range(num_epochs):
        print(f"\n🔁 Epoch {epoch+1}/{num_epochs}")
        model.train()
        train_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = 100 * correct / total
        print(f"✅ Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_acc = 100 * correct / total
        print(f"🧪 Val Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")

    return model
trained_model = train_model(model, criterion, optimizer, train_loader, val_loader, device, num_epochs=5)


In [ ]:
torch.save(trained_model.state_dict(), "mobilenet_waste_classifier.pth")
print("✅ Model saved as mobilenet_waste_classifier.pth")
model.load_state_dict(torch.load("mobilenet_waste_classifier.pth"))
model.eval()
import matplotlib.pyplot as plt

def predict_image(image_path, model, transform, label_map):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    img_tensor = transform(image).unsqueeze(0)  # Add batch dimension

    with torch.no_grad():
        outputs = model(img_tensor.to(device))
        _, predicted = torch.max(outputs, 1)

    predicted_class = list(label_map.keys())[list(label_map.values()).index(predicted.item())]

    # Show image + prediction
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Predicted: {predicted_class}", fontsize=14)
    plt.show()
predict_image("train_data/train/metal249_jpg.rf.1e25d765cca05bdbdcff64c46fff6365.jpg", trained_model, transform, label_to_index)



In [ ]:
import random

sample_images = df["filepath"].sample(5, random_state=42).tolist()
for path in sample_images:
    print("📷", path)


In [ ]:
predict_image("MYTEST2.jpg", trained_model, transform, label_to_index)


In [ ]:
predict_image("MYTEST3.jpg", trained_model, transform, label_to_index)


In [ ]:
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def preprocess_and_predict(image_path, model, transform, label_map):
    model.eval()

    # Step 1: Load with OpenCV
    original = cv2.imread(image_path)
    if original is None:
        print("❌ Could not load image.")
        return
    img = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)

    # Step 2: Apply image processing (you can try different techniques here)
    img_blur = cv2.GaussianBlur(img, (5, 5), 0)
    img_thresh = cv2.adaptiveThreshold(img_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 11, 2)

    # Optional: show the processed image
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(img_thresh, cmap="gray")
    plt.title("Processed (Threshold)")
    plt.axis("off")
    plt.show()

    # Step 3: Convert processed image to RGB and PIL for model input
    img_rgb = cv2.cvtColor(img_thresh, cv2.COLOR_GRAY2RGB)
    pil_image = Image.fromarray(img_rgb)

    # Step 4: Apply model transforms
    img_tensor = transform(pil_image).unsqueeze(0).to(device)

    # Step 5: Predict
    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = torch.max(outputs, 1)

    predicted_class = list(label_map.keys())[list(label_map.values()).index(predicted.item())]

    print(f"🔮 Predicted Class: {predicted_class}")
    
preprocess_and_predict("MYTEST2.jpg", trained_model, transform, label_to_index)


In [ ]:
predict_image("metal.jpg", trained_model, transform, label_to_index)

In [ ]:
predict_image("papper.jpg", trained_model, transform, label_to_index)

In [ ]:
predict_image("glass.jpg", trained_model, transform, label_to_index)

In [ ]:
predict_image("plastic2.jpg", trained_model, transform, label_to_index)

In [ ]:
predict_image("glass2.jpg", trained_model, transform, label_to_index)

In [ ]:
predict_image("glass3.jpg", trained_model, transform, label_to_index)

In [ ]:
predict_image("metal2.jpg", trained_model, transform, label_to_index)

In [ ]:
predict_image("metal3.jpg", trained_model, transform, label_to_index)

In [ ]:
predict_image("paper2.jpg", trained_model, transform, label_to_index)

In [ ]:
predict_image("paper3.jpg", trained_model, transform, label_to_index)

In [ ]:
import pandas as pd

# Define the hard sample image paths and labels
data = {
    "filepath": [
        "/mnt/data/plastic2.jpg",
        "/mnt/data/metal.jpg",
        "/mnt/data/glass.jpg",
        "/mnt/data/glass3.jpg",
        "/mnt/data/papper.jpg",
        "/mnt/data/paper2.jpg"
    ],
    "label": [
        "plastic",
        "metal",
        "glass",
        "glass",
        "paper",
        "paper"
    ]
}

hard_df = pd.DataFrame(data)

# Encode labels
label_to_index = {'glass': 0, 'metal': 1, 'paper': 2, 'plastic': 3}
hard_df["label_encoded"] = hard_df["label"].map(label_to_index)

hard_df.to_csv("hard_samples.csv", index=False)
print("✅ Created hard_samples.csv")


In [ ]:
# Load hard_samples.csv
import pandas as pd

hard_df = pd.read_csv("hard_samples.csv")
print("✅ Loaded hard sample dataset")
print(hard_df.head())


In [ ]:
# Use your existing transform
hard_dataset = WasteDataset(hard_df, transform=transform)

# Create a DataLoader
from torch.utils.data import DataLoader

hard_loader = DataLoader(hard_dataset, batch_size=4, shuffle=True)


In [ ]:
import pandas as pd

# Set relative paths to your hard_samples folder
hard_df = pd.DataFrame({
    "filepath": [
        "hard_samples/glass.jpg",
        "hard_samples/glass3.jpg",
        "hard_samples/metal.jpg",
        "hard_samples/paper2.jpg",
        "hard_samples/papper.jpg"
    ],
    "label": [
        "glass",
        "glass",
        "metal",
        "paper",
        "paper"
    ]
})

# Encode labels
label_to_index = {'glass': 0, 'metal': 1, 'paper': 2, 'plastic': 3}
hard_df["label_encoded"] = hard_df["label"].map(label_to_index)

# Save to CSV
hard_df.to_csv("hard_samples.csv", index=False)
print("✅ hard_samples.csv created")


In [ ]:
from torch.utils.data import DataLoader

# Load the hard sample dataframe
hard_df = pd.read_csv("hard_samples.csv")

# Reuse your existing WasteDataset class
hard_dataset = WasteDataset(hard_df, transform=transform)

# Create a DataLoader for the hard sample dataset
hard_loader = DataLoader(hard_dataset, batch_size=4, shuffle=True)

import torch
import torch.nn as nn
import torch.optim as optim

# Set model to training mode
model.train()

# Use a low learning rate to avoid overwriting the original model
optimizer = optim.Adam(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

# Fine-tune for 3 epochs (you can increase later)
for epoch in range(3):
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in hard_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"🔁 Epoch {epoch+1} | Loss: {total_loss:.4f} | Accuracy on hard set: {acc:.2f}%")


In [ ]:
import urllib.request
import zipfile
import os

url = "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"
local_zip = "trashnet.zip"

if not os.path.exists(local_zip):
    print("📥 Downloading TrashNet...")
    urllib.request.urlretrieve(url, local_zip)
    print("✅ Downloaded.")

extract_folder = "trashnet_data"
with zipfile.ZipFile(local_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)
    print(f"✅ Extracted to {extract_folder}")


In [ ]:
import pandas as pd
import os

# Define classes you want
selected_classes = ['glass', 'metal', 'paper', 'plastic']

# Path to dataset
dataset_dir = "trashnet_data/dataset-resized"

# Collect image paths and labels
filepaths = []
labels = []

for class_name in selected_classes:
    class_dir = os.path.join(dataset_dir, class_name)
    if os.path.isdir(class_dir):
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                filepaths.append(os.path.join(class_dir, fname))
                labels.append(class_name)

# Create DataFrame
df_trash = pd.DataFrame({'filepath': filepaths, 'label': labels})

# Encode labels
label_to_index = {label: idx for idx, label in enumerate(sorted(selected_classes))}
df_trash['label_encoded'] = df_trash['label'].map(label_to_index)

print(f"✅ Found {len(df_trash)} images across {len(selected_classes)} classes.")
df_trash.head()


In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_test = train_test_split(
    df_trash, test_size=0.2, stratify=df_trash["label_encoded"], random_state=42
)

print(f"🧠 Train size: {len(df_train)}")
print(f"🧪 Test size: {len(df_test)}")


In [ ]:
# Create training and test datasets
train_dataset = WasteDataset(df_train, transform=transform)
test_dataset = WasteDataset(df_test, transform=transform)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print(f"✅ Ready with {len(train_dataset)} train and {len(test_dataset)} test images.")

trained_model = train_model(model, criterion, optimizer, train_loader, val_loader=test_loader, device=device, num_epochs=5)


In [ ]:
torch.save(trained_model.state_dict(), "mobilenet_trashnet_4class.pth")
print("💾 Model saved as mobilenet_trashnet_4class.pth")


In [ ]:
predict_image("MYTEST2.jpg", trained_model, transform, label_to_index)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

def enhanced_preprocess_and_predict(image_path, model, transform, label_map):
    model.eval()

    # Load image with OpenCV
    img = cv2.imread(image_path)
    if img is None:
        print("❌ Image not found.")
        return

    original = img.copy()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 1️⃣ CLAHE (adaptive histogram equalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    clahe_img = clahe.apply(gray)

    # 2️⃣ Detect & inpaint glare regions
    _, glare_mask = cv2.threshold(clahe_img, 240, 255, cv2.THRESH_BINARY)
    inpainted = cv2.inpaint(img, glare_mask, 3, cv2.INPAINT_TELEA)

    # 3️⃣ Background removal with adaptive threshold
    gray_inpainted = cv2.cvtColor(inpainted, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray_inpainted, (5, 5), 0)
    mask = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                 cv2.THRESH_BINARY_INV, 11, 3)
    cleaned = cv2.bitwise_and(inpainted, inpainted, mask=mask)

    # 4️⃣ Convert to RGB → PIL → Tensor
    final_rgb = cv2.cvtColor(cleaned, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(final_rgb)
    img_tensor = transform(pil_img).unsqueeze(0).to(model.device if hasattr(model, "device") else "cpu")

    # Prediction
    with torch.no_grad():
        outputs = model(img_tensor)
        _, pred = torch.max(outputs, 1)
        pred_class = list(label_map.keys())[list(label_map.values()).index(pred.item())]

    # Visualization
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(cv2.cvtColor(inpainted, cv2.COLOR_BGR2RGB))
    plt.title("Inpainted (Glare Removed)")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(final_rgb)
    plt.title(f"Preprocessed → Predicted: {pred_class}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()
enhanced_preprocess_and_predict("MYTEST2.jpg", trained_model, transform, label_to_index)


In [ ]:
predict_image("paper2.jpg", trained_model, transform, label_to_index)

In [ ]:
import zipfile
import os

zip_path = "archive.zip"
extract_to = "waste_dataset"

# Extract the ZIP
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("✅ Dataset extracted to:", extract_to)


In [ ]:
import os

# See what subfolders are inside
print("📂 Subfolders inside extracted dataset:")
for folder in os.listdir(extract_to):
    print(" -", folder)


In [ ]:
import os

folder_path = "waste_dataset/Garbage classification"

print("📁 Class folders inside 'Garbage classification':")
for folder in os.listdir(folder_path):
    print("-", folder)


In [ ]:
inner_path = "waste_dataset/Garbage classification/Garbage classification"

print("📁 Class folders inside inner folder:")
for folder in os.listdir(inner_path):
    print("-", folder)



In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Path to the useful data
source_dir = "waste_dataset/Garbage classification/Garbage classification"
target_classes = ["glass", "metal", "paper", "plastic"]

# Create destination folders
train_dir = "waste_dataset/cleaned_split/train"
test_dir = "waste_dataset/cleaned_split/test"

os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Loop over each class
for class_name in target_classes:
    class_path = os.path.join(source_dir, class_name)
    images = os.listdir(class_path)

    # Split
    train_imgs, test_imgs = train_test_split(images, test_size=0.2, random_state=42)

    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(test_dir, class_name), exist_ok=True)

    for img in train_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(train_dir, class_name, img)
        shutil.copy2(src, dst)

    for img in test_imgs:
        src = os.path.join(class_path, img)
        dst = os.path.join(test_dir, class_name, img)
        shutil.copy2(src, dst)

print("✅ Data filtered and split into:")
print("📁", train_dir)
print("📁", test_dir)


In [ ]:
import os

print("📁 Listing contents of train_data:")
print(os.listdir("train_data"))


In [ ]:
import os

print("📁 Class folders inside train_data/train:")
print(os.listdir("train_data/train"))


In [ ]:
import os
import shutil

# المسار إلى مجلد الصور
base_path = "train_data/train"

# إنشاء مجلدات للفئات تلقائيًا إذا لم تكن موجودة
classes = ["glass", "plastic", "paper", "metal"]
for cls in classes:
    os.makedirs(os.path.join(base_path, cls), exist_ok=True)

# نقل كل صورة إلى مجلدها المناسب حسب اسمها
for filename in os.listdir(base_path):
    if filename.endswith(".jpg") or filename.endswith(".png"):
        for cls in classes:
            if filename.lower().startswith(cls):  # مثل glass1_...
                src_path = os.path.join(base_path, filename)
                dst_path = os.path.join(base_path, cls, filename)
                shutil.move(src_path, dst_path)
                break  # خلاص وجدنا الفئة المناسبة، نوقف


In [ ]:
import shutil
import os

train_path = "train_data/train"
# حذف أي مجلد غير من مجلدات التصنيفات الصحيحة
for item in os.listdir(train_path):
    item_path = os.path.join(train_path, item)
    if item.startswith('.') or item.endswith('.csv'):
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
        else:
            os.remove(item_path)


In [ ]:
import os

print("📂 المجلدات في مجلد المشروع الحالي:")
print(os.listdir())


In [ ]:
import os

print("📁 Contents of trashnet_data:")
print(os.listdir("trashnet_data"))


In [ ]:
path_trashnet_train = "trashnet_data/dataset-resized"
dataset_trashnet = ImageFolder(path_trashnet_train, transform=transform)


In [ ]:
from torchvision.datasets import ImageFolder
from torchvision import transforms

# Define the transformation (you can modify it as needed)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Set correct paths based on your current directory structure
path_original_train = "train_data/train"
path_trashnet_train  = "trashnet_data/dataset-resized"
path_garbage_train   = "waste_dataset/cleaned_split/train"

# Load each train dataset
dataset_original = ImageFolder(path_original_train, transform=transform)
dataset_trashnet = ImageFolder(path_trashnet_train, transform=transform)
dataset_garbage  = ImageFolder(path_garbage_train, transform=transform)


In [ ]:
from torch.utils.data import ConcatDataset

# Combine datasets
combined_dataset = ConcatDataset([dataset_original, dataset_trashnet, dataset_garbage])


In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(combined_dataset))
val_size = len(combined_dataset) - train_size

train_dataset, val_dataset = random_split(combined_dataset, [train_size, val_size])


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset

allowed_classes = ['glass', 'metal', 'paper', 'plastic']
class_to_index = {cls: i for i, cls in enumerate(allowed_classes)}

def load_and_filter_dataset(path, transform):
    dataset = ImageFolder(path, transform=transform)
    allowed_indices = []
    new_samples = []

    for idx, (img_path, label) in enumerate(dataset.samples):
        class_name = dataset.classes[label]
        if class_name in allowed_classes:
            new_label = class_to_index[class_name]
            new_samples.append((img_path, new_label))
            allowed_indices.append(idx)

    dataset.samples = new_samples
    dataset.targets = [label for _, label in new_samples]
    dataset.classes = allowed_classes
    dataset.class_to_idx = class_to_index
    return dataset


In [ ]:
dataset_original = load_and_filter_dataset(path_original_train, transform)
dataset_trashnet = load_and_filter_dataset(path_trashnet_train, transform)
dataset_garbage  = load_and_filter_dataset(path_garbage_train, transform)


In [ ]:
print("Original dataset classes:", dataset_original.classes)
print("TrashNet dataset classes:", dataset_trashnet.classes)
print("Garbage dataset classes:", dataset_garbage.classes)

print("Unique labels in combined dataset:")
all_labels = [label for _, label in dataset_original.samples + dataset_trashnet.samples + dataset_garbage.samples]
print(sorted(set(all_labels)))  # Should print [0, 1, 2, 3]


In [ ]:
sample_batch = next(iter(train_loader))
images, labels = sample_batch
print("Sample batch labels:", labels.tolist())

if max(labels) > 3:
    print("❌ ERROR: Label out of range!")
else:
    print("✅ All labels are within correct range (0–3)")


In [ ]:
num_epochs = 1


In [ ]:
# Only allow these 4 classes
allowed_classes = ['glass', 'metal', 'paper', 'plastic']
allowed_class_to_idx = {cls: i for i, cls in enumerate(allowed_classes)}

# Custom filtered dataset for TrashNet
def filter_trashnet(dataset):
    filtered_samples = [(path, allowed_class_to_idx[cls])
                        for path, cls_idx in dataset.samples
                        if dataset.classes[cls_idx] in allowed_class_to_idx]
    dataset.samples = filtered_samples
    dataset.targets = [label for _, label in filtered_samples]
    dataset.classes = allowed_classes
    dataset.class_to_idx = allowed_class_to_idx
    return dataset

# Apply the filter
dataset_trashnet = filter_trashnet(dataset_trashnet)


In [ ]:
print("Original dataset classes:", dataset_original.classes)
print("TrashNet dataset classes:", dataset_trashnet.classes)
print("Garbage dataset classes:", dataset_garbage.classes)


In [ ]:
# Check the number of output classes in the model
print("🧠 Model output classes:", model.classifier[1].out_features)

# Check labels in train_loader
for _, labels in train_loader:
    print("🎯 Labels in batch:", labels.unique())
    break


In [ ]:
def filter_trashnet(dataset):
    allowed_classes = ['glass', 'metal', 'paper', 'plastic']
    allowed_class_to_idx = {cls: i for i, cls in enumerate(allowed_classes)}

    filtered = []
    for path, label in dataset.samples:
        class_name = dataset.classes[label]
        if class_name in allowed_classes:
            new_label = allowed_class_to_idx[class_name]
            filtered.append((path, new_label))

    dataset.samples = filtered
    dataset.targets = [label for _, label in filtered]
    dataset.classes = allowed_classes
    dataset.class_to_idx = allowed_class_to_idx
    return dataset

dataset_trashnet = filter_trashnet(dataset_trashnet)


In [ ]:
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)


In [ ]:
def filter_and_remap_trashnet(dataset):
    # Keep only these 4
    allowed_classes = ['glass', 'metal', 'paper', 'plastic']
    allowed_class_to_idx = {cls: i for i, cls in enumerate(allowed_classes)}

    new_samples = []
    for path, label in dataset.samples:
        original_class = dataset.classes[label]
        if original_class in allowed_classes:
            new_label = allowed_class_to_idx[original_class]
            new_samples.append((path, new_label))

    dataset.samples = new_samples
    dataset.targets = [label for _, label in new_samples]
    dataset.classes = allowed_classes
    dataset.class_to_idx = allowed_class_to_idx
    return dataset

# Apply it again
dataset_trashnet = filter_and_remap_trashnet(dataset_trashnet)


In [ ]:
for _, labels in train_loader:
    print("🧪 Labels in batch after fix:", labels.unique())
    break


In [ ]:
from torchvision.datasets import ImageFolder

# 🛠️ Fix function to clean TrashNet dataset
def filter_and_remap_trashnet(dataset):
    allowed_classes = ['glass', 'metal', 'paper', 'plastic']
    allowed_class_to_idx = {cls: i for i, cls in enumerate(allowed_classes)}

    new_samples = []
    for path, label in dataset.samples:
        original_class = dataset.classes[label]
        if original_class in allowed_classes:
            new_label = allowed_class_to_idx[original_class]
            new_samples.append((path, new_label))

    dataset.samples = new_samples
    dataset.targets = [label for _, label in new_samples]
    dataset.classes = allowed_classes
    dataset.class_to_idx = allowed_class_to_idx
    return dataset

# ✅ Apply the fix on TrashNet
filtered_trashnet_labels = None
try:
    dataset_trashnet = ImageFolder("trashnet_data/dataset-resized", transform=None)
    dataset_trashnet = filter_and_remap_trashnet(dataset_trashnet)
    filtered_trashnet_labels = set(label for _, label in dataset_trashnet.samples)
except Exception as e:
    filtered_trashnet_labels = str(e)

filtered_trashnet_labels


In [ ]:
# Apply transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Reload Original and Garbage datasets with transform
dataset_original = ImageFolder("train_data/train", transform=transform)
dataset_garbage = ImageFolder("waste_dataset/cleaned_split/train", transform=transform)

# TrashNet is already filtered and loaded
dataset_trashnet.transform = transform  # 🔧 Attach transform to the filtered one

# Combine datasets
combined_dataset = ConcatDataset([dataset_original, dataset_trashnet, dataset_garbage])


In [ ]:
from torch.utils.data import random_split, DataLoader

train_size = int(0.8 * len(combined_dataset))
val_size = len(combined_dataset) - train_size
train_dataset, val_dataset = random_split(combined_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


In [ ]:
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

# Load MobileNetV2 and adjust classifier
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
for param in model.features.parameters():
    param.requires_grad = False
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)
model = model.to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss:.4f}, Accuracy: {100*correct/total:.2f}%")

# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

print(f"✅ Final Validation Accuracy: {100 * correct / total:.2f}%")


In [ ]:
torch.save(model.state_dict(), "mobilenet_waste_classifier.pth")
print("✅ Model saved.")


In [ ]:
from PIL import Image
import torchvision.transforms as T

# Load class names (make sure they match order)
class_names = ['glass', 'metal', 'paper', 'plastic']

# Load and preprocess image
img_path = "MYTEST2.jpg"
image = Image.open(img_path).convert("RGB")
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])
input_tensor = transform(image).unsqueeze(0).to(device)

# Predict
model.eval()
with torch.no_grad():
    output = model(input_tensor)
    predicted = output.argmax(dim=1).item()
    print(f"🧠 Prediction: {class_names[predicted]}")


In [ ]:
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt

# Load class names
class_names = ['glass', 'metal', 'paper', 'plastic']

# Load and preprocess image
img_path = "MYTEST2.jpg"
image = Image.open(img_path).convert("RGB")
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])
input_tensor = transform(image).unsqueeze(0).to(device)

# Predict
model.eval()
with torch.no_grad():
    output = model(input_tensor)
    predicted = output.argmax(dim=1).item()
    predicted_label = class_names[predicted]

# Show image with prediction
plt.imshow(image)
plt.title(f"🧠 Prediction: {predicted_label}", fontsize=14)
plt.axis('off')
plt.show()


In [ ]:
predict_image("paper2.jpg", trained_model, transform, label_to_index)

In [ ]:
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt

# Load class names
class_names = ['glass', 'metal', 'paper', 'plastic']

# Load and preprocess image
img_path = "paper2.jpg"
image = Image.open(img_path).convert("RGB")
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])
input_tensor = transform(image).unsqueeze(0).to(device)

# Predict
model.eval()
with torch.no_grad():
    output = model(input_tensor)
    predicted = output.argmax(dim=1).item()
    predicted_label = class_names[predicted]

# Show image with prediction
plt.imshow(image)
plt.title(f"🧠 Prediction: {predicted_label}", fontsize=14)
plt.axis('off')
plt.show()


In [ ]:
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt
import torch
import os

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Class names
class_names = ['glass', 'metal', 'paper', 'plastic']

# Image preprocessing transform
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
])

# List of test image paths
image_paths = [
    "MYTEST2.jpg",
    "paper2.jpg",
    # Add more image paths here if needed
]

# Set model to evaluation mode
model.eval()

# Plot images and predictions
plt.figure(figsize=(12, 4))
for i, img_path in enumerate(image_paths):
    # Load and preprocess image
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        output = model(input_tensor)
        predicted = output.argmax(dim=1).item()
        predicted_label = class_names[predicted]

    # Show image
    plt.subplot(1, len(image_paths), i + 1)
    plt.imshow(image)
    plt.title(f"Prediction: {predicted_label}")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# 🧠 Predict class
model.eval()
with torch.no_grad():
    output = model(input_tensor)
    predicted = output.argmax(dim=1).item()
    confidence = torch.nn.functional.softmax(output, dim=1)[0][predicted].item()

# ✅ Visualize image with prediction and confidence
plt.figure(figsize=(4, 4))
plt.imshow(image)
plt.title(f"Prediction: {class_names[predicted]} ({confidence * 100:.2f}%)")
plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
from tqdm import tqdm

model.eval()  # Set the model to evaluation mode
correct = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\n✅ Final Validation Accuracy: {accuracy:.2f}%")


In [ ]:
from tqdm import tqdm

model.eval()  # Set the model to evaluation mode
correct = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\n✅ Final Validation Accuracy: {accuracy:.2f}%")


In [ ]:
import torch
import os
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "hard_samples"  # Make sure this path is correct

# ✅ Define transform (same as during training)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ✅ Load test images
image_files = [f for f in os.listdir(test_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# ✅ Set model to eval mode
model.eval()

# ✅ Plot predictions
plt.figure(figsize=(len(image_files) * 3.5, 4))
for idx, fname in enumerate(image_files):
    img_path = os.path.join(test_folder, fname)
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        predicted_label = class_names[probs.argmax().item()]
        confidence = probs.max().item() * 100

    plt.subplot(1, len(image_files), idx + 1)
    plt.imshow(image)
    plt.title(f"{predicted_label}\n({confidence:.1f}%)")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:


import torch
import torchvision.models as models
torch.save(model.state_dict(), "mobilenet_waste_classifier.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Create a new MobileNetV2 model
model = models.mobilenet_v2(pretrained=True)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 4)
model.load_state_dict(torch.load("mobilenet_waste_classifier.pth"))
model = model.to(device)
model.eval()

print("✅ Model loaded and ready.")


In [ ]:
# Use the same val_loader you had before
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

accuracy = 100 * correct / total
print(f"✅ Final Validation Accuracy: {accuracy:.2f}%")


In [ ]:
import os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import transforms, models, datasets
from torch.utils.data import DataLoader

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "hard_samples"  # Make sure this folder contains class-named subfolders

# ✅ Transform (same as training)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ✅ Load trained model
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)
model.load_state_dict(torch.load("mobilenet_waste_classifier.pth", map_location=device))
model = model.to(device)
model.eval()

# ✅ Load test dataset
test_dataset = datasets.ImageFolder(test_folder, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# ✅ Plot + Accuracy
plt.figure(figsize=(len(test_dataset) * 4, 4))
correct = 0
total = 0

for idx, (image, label) in enumerate(test_loader):
    image = image.to(device)
    label = label.item()
    with torch.no_grad():
        output = model(image)
        probs = torch.softmax(output, dim=1)
        predicted_idx = probs.argmax().item()
        predicted_label = class_names[predicted_idx]
        confidence = probs[0, predicted_idx].item() * 100

    total += 1
    correct += (predicted_idx == label)

    # Show image and highlight wrong predictions in red
    title_color = "green" if predicted_idx == label else "red"
    plt.subplot(1, len(test_dataset), idx + 1)
    plt.imshow(image.cpu().squeeze().permute(1, 2, 0))
    plt.title(f"{predicted_label}\n({confidence:.1f}%)\nGT: {class_names[label]}", color=title_color)
    plt.axis("off")

plt.tight_layout()
plt.show()

# ✅ Accuracy summary
accuracy = 100 * correct / total
print(f"\n🎯 Test Accuracy on {total} images: {accuracy:.2f}%")


In [ ]:
import os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import transforms, models, datasets
from torch.utils.data import DataLoader

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "hard_samples"

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ✅ Load model
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)
model.load_state_dict(torch.load("mobilenet_waste_classifier.pth", map_location=device))
model = model.to(device)
model.eval()

# ✅ Load dataset
test_dataset = datasets.ImageFolder(test_folder, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# ✅ Plot setup
cols = 4
rows = (len(test_dataset) + cols - 1) // cols
fig = plt.figure(figsize=(cols * 4, rows * 4))

correct = 0
total = 0

# ✅ Prediction loop
for idx, (image, label) in enumerate(test_loader):
    image = image.to(device)
    label = label.item()
    with torch.no_grad():
        output = model(image)
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax().item()
        confidence = probs[0, pred_idx].item() * 100

    correct += (pred_idx == label)
    total += 1

    ax = fig.add_subplot(rows, cols, idx + 1)
    ax.imshow(image.cpu().squeeze().permute(1, 2, 0))
    ax.axis("off")
    ax.set_title(f"GT: {class_names[label]}\nPred: {class_names[pred_idx]} ({confidence:.1f}%)", 
                 color="green" if pred_idx == label else "red")

plt.tight_layout()
plt.show()

# ✅ Final Accuracy
print(f"✅ Final Accuracy: {100 * correct / total:.2f}%")


In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "hard_samples"  # just images inside, no subfolders

# ✅ Preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ✅ Load model
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)
model.load_state_dict(torch.load("mobilenet_waste_classifier.pth", map_location=device))
model = model.to(device)
model.eval()

# ✅ Load image paths
image_files = [f for f in os.listdir(test_folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
image_files = image_files[:5]  # only take first 5 images

# ✅ Plot
plt.figure(figsize=(20, 5))

for idx, fname in enumerate(image_files):
    path = os.path.join(test_folder, fname)
    img = Image.open(path).convert("RGB")
    input_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred = probs.argmax().item()
        conf = probs[0, pred].item() * 100

    # Show image
    plt.subplot(1, 5, idx + 1)
    plt.imshow(img)
    plt.title(f"Pred: {class_names[pred]}\n({conf:.1f}%)")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import os
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# ✅ إعداد البيئة
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "hard_samples"  # تأكد أن هذا المسار صحيح

# ✅ التحويلات للصورة
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ✅ تحميل النموذج
from torchvision.models import mobilenet_v2
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, len(class_names))
model.load_state_dict(torch.load("mobilenet_waste_classifier.pth", map_location=device))
model = model.to(device)
model.eval()

# ✅ استخراج الصور والمسميات Ground Truth من أسماء الملفات
image_files = [f for f in os.listdir(test_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
image_paths = [os.path.join(test_folder, f) for f in image_files]
true_labels = [next((i for i, cls in enumerate(class_names) if cls in f.lower()), None) for f in image_files]

# ✅ عرض الصور والتنبؤات
plt.figure(figsize=(len(image_files) * 4, 5))

for idx, (img_path, true_label) in enumerate(zip(image_paths, true_labels)):
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        predicted_index = probs.argmax().item()
        predicted_label = class_names[predicted_index]
        confidence = probs.max().item() * 100

    # ✅ مقارنة النتيجة مع الحقيقة
    is_correct = predicted_index == true_label
    title_color = "green" if is_correct else "red"
    true_cls_name = class_names[true_label] if true_label is not None else "?"

    plt.subplot(1, len(image_files), idx + 1)
    plt.imshow(image)
    plt.title(f"GT: {true_cls_name}\nPred: {predicted_label} ({confidence:.1f}%)", color=title_color)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
torch.save(model.state_dict(), "mobilenet_original_95_14.pth")

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([ transforms.RandomResizedCrop(224, scale=(0.8, 1.0)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(degrees=15), transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), transforms.ToTensor(), ])

val_transform = transforms.Compose([ transforms.Resize((224, 224)), transforms.ToTensor(), ])

In [ ]:
for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    _, predicted = outputs.max(1)
    correct += predicted.eq(labels).sum().item()
    total += labels.size(0)

acc = 100 * correct / total
print(f"📘 Epoch {epoch+1}/{num_epochs} | Loss: {total_loss:.2f} | Accuracy: {acc:.2f}%")


In [ ]:
import torch

# Set model to evaluation mode
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

accuracy = 100 * correct / total
print(f"\n✅ Final Validation Accuracy: {accuracy:.2f}%")


In [ ]:
for batch in val_loader:
    print("✅ Validation batch loaded")
    break

In [ ]:
from PIL import Image
from torchvision import transforms
import torch
import matplotlib.pyplot as plt
import os

# Update as needed
image_folder = 'hard_samples'  # or your test folder
class_names = ['glass', 'metal', 'paper', 'plastic']

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

model.eval()
plt.figure(figsize=(len(os.listdir(image_folder)) * 4, 4))

for idx, file in enumerate(os.listdir(image_folder)):
    if file.lower().endswith(('jpg', 'png', 'jpeg')):
        path = os.path.join(image_folder, file)
        image = Image.open(path).convert("RGB")
        input_tensor = val_transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            probs = torch.softmax(output, dim=1)
            pred_idx = probs.argmax().item()
            confidence = probs.max().item() * 100

        plt.subplot(1, len(os.listdir(image_folder)), idx + 1)
        plt.imshow(image)
        plt.title(f"{class_names[pred_idx]}\n({confidence:.1f}%)")
        plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:


import torch
from torchvision import transforms
from PIL import Image
import os
import matplotlib.pyplot as plt

# ✅ Device and class names
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Load images from folder
test_folder = "hard_samples"
image_files = [f for f in os.listdir(test_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# ✅ Model in eval mode
model.eval()

# ✅ Create figure
plt.figure(figsize=(len(image_files) * 4, 4))

for idx, file in enumerate(image_files):
    # Load image
    path = os.path.join(test_folder, file)
    image = Image.open(path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    # Predict
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        predicted_label = class_names[probs.argmax().item()]
        confidence = probs.max().item() * 100

    # Get GT label from filename
    if "_" in file:
        gt_label = file.split("_")[0].lower()
    else:
        gt_label = file.split(".")[0].lower()

    # Determine color
    is_correct = (predicted_label.lower() == gt_label)
    title_color = "green" if is_correct else "red"

    # Show image
    plt.subplot(1, len(image_files), idx + 1)
    plt.imshow(image)
    plt.title(f"GT: {gt_label}\nPred: {predicted_label} ({confidence:.1f}%)", color=title_color)
    plt.axis("off")

plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd

# Assuming 'misclassified' list is already created
df_misclassified = pd.DataFrame(misclassified, columns=["Filename", "True Label", "Predicted Label", "Confidence"])
print("Misclassified Samples:")
print(df_misclassified)

# If you want a nice table in Jupyter
df_misclassified.head(10)  # shows the first 10 rows


In [ ]:


import os
import shutil
import pandas as pd

# Assuming df_misclassified already exists and contains a column "Filename"
source_folder = "hard_samples"
target_folder = "hard_finetune"

# Create the target directory if it doesn't exist
os.makedirs(target_folder, exist_ok=True)

# Loop through filenames and copy each one
for filename in df_misclassified["Filename"]:
    src = os.path.join(source_folder, filename)
    dst = os.path.join(target_folder, filename)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"✅ Copied: {filename}")
    else:
        print(f"❌ Not Found: {filename}")

print("\n🎯 Done! Misclassified images copied to:", target_folder)



In [ ]:
import os
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms
import shutil

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "hard_samples"

# Image transform must match validation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

misclassified = []

# Loop through images
for fname in os.listdir(test_folder):
    if fname.lower().endswith((".jpg", ".png", ".jpeg")):
        path = os.path.join(test_folder, fname)
        image = Image.open(path).convert("RGB")
        input_tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            probs = torch.softmax(output, dim=1)
            pred_idx = probs.argmax(dim=1).item()
            pred_label = class_names[pred_idx]
            confidence = probs.max().item() * 100

        # Get ground truth label from filename (e.g., 'paper_2.jpg' → 'paper')
        true_label = fname.split("_")[0].lower()
        if true_label != pred_label:
            misclassified.append([fname, true_label, pred_label, confidence])

# Save misclassified images to hard_finetune folder
os.makedirs("hard_finetune", exist_ok=True)

for fname, true, pred, conf in misclassified:
    shutil.copy2(os.path.join(test_folder, fname), os.path.join("hard_finetune", fname))

# Show results as a table
df_misclassified = pd.DataFrame(misclassified, columns=["Filename", "True Label", "Predicted Label", "Confidence"])
df_misclassified.head()


In [ ]:
# Updated function to extract true label from filename
def extract_true_label(filename):
    return filename.split("_")[0].split(".")[0]  # handles 'glass.jpg', 'glass_3.jpg'

misclassified = []

# Inference loop
for fname in os.listdir(test_folder):
    if not fname.lower().endswith((".jpg", ".png", ".jpeg")):
        continue

    img_path = os.path.join(test_folder, fname)
    image = Image.open(img_path).convert("RGB")
    input_tensor = val_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probs = torch.softmax(outputs, dim=1)
        predicted_idx = probs.argmax().item()
        predicted_label = class_names[predicted_idx]
        confidence = probs.max().item() * 100

    true_label = extract_true_label(fname)

    if predicted_label != true_label:
        misclassified.append([fname, true_label, predicted_label, confidence])


In [ ]:
import pandas as pd

# Create DataFrame from misclassified list
df_misclassified = pd.DataFrame(misclassified, columns=["Filename", "True Label", "Predicted Label", "Confidence"])

# Display the DataFrame
from IPython.display import display
display(df_misclassified)


In [ ]:
import os
import shutil

# Folder to save hard examples
hard_folder = "hard_finetune"
os.makedirs(hard_folder, exist_ok=True)

copied_files = []

# Go through the cleaned DataFrame and copy files
for fname, true_label, pred_label, conf in df_misclassified.itertuples(index=False):
    src_path = os.path.join("hard_samples", fname)
    dst_path = os.path.join(hard_folder, fname)

    if os.path.exists(src_path) and fname not in copied_files:
        shutil.copy(src_path, dst_path)
        copied_files.append(fname)
        print(f"✅ Copied: {fname}")

print(f"\n🎯 Done! {len(copied_files)} misclassified images copied to: {hard_folder}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms
import torchvision.models as models

# ✅ Set device and parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hard_data_path = "hard_finetune"
num_epochs = 5
batch_size = 8
learning_rate = 1e-4

# ✅ Load your already trained model (use your best model .pth path)
model = models.mobilenet_v2(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)
model.load_state_dict(torch.load("mobilenet_waste_classifier.pth"))  # <- change if needed
model.to(device)

# ✅ Freeze all except last few layers
for name, param in model.features.named_parameters():
    if "18" not in name:  # unfreeze last block only
        param.requires_grad = False

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Dataset and DataLoader
dataset = ImageFolder(hard_data_path, transform=transform)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# ✅ Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)

# ✅ Fine-tuning loop
model.train()
for epoch in range(num_epochs):
    running_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    print(f"📘 Epoch {epoch+1}/{num_epochs}, Loss: {running_loss:.4f}, Accuracy: {100 * correct / total:.2f}%")

# ✅ Save updated model
torch.save(model.state_dict(), "mobilenet_finetuned.pth")
print("\n✅ Model fine-tuned and saved as mobilenet_finetuned.pth")


In [ ]:
import os
import shutil

# Organize images into subfolders based on true labels in filename
source_folder = "hard_finetune"
organized = 0

for file in os.listdir(source_folder):
    if file.lower().endswith((".jpg", ".jpeg", ".png")):
        lower_file = file.lower()
        if "glass" in lower_file:
            class_name = "glass"
        elif "paper" in lower_file:
            class_name = "paper"
        elif "metal" in lower_file:
            class_name = "metal"
        elif "plastic" in lower_file:
            class_name = "plastic"
        else:
            continue  # Skip unknowns

        # Create class folder if it doesn't exist
        class_folder = os.path.join(source_folder, class_name)
        os.makedirs(class_folder, exist_ok=True)

        # Move image into the correct class folder
        src_path = os.path.join(source_folder, file)
        dst_path = os.path.join(class_folder, file)
        shutil.move(src_path, dst_path)
        organized += 1

print(f"✅ Organized {organized} image(s) into class folders inside '{source_folder}'")


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F

# ✅ Step 1: Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Step 2: Validation transform (same as before)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Step 3: Load validation dataset (corrected path)
val_dir = "waste_dataset/cleaned_split/test"
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

# ✅ Step 4: Load your trained model
from torchvision import models
import torch.nn as nn

model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(model.last_channel, 4)  # 4 classes
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model = model.to(device)
model.eval()

# ✅ Step 5: Evaluate on validation set
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

accuracy = 100 * correct / total
print(f"\n✅ Final Validation Accuracy: {accuracy:.2f}%")


In [ ]:

import torch
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Rebuild MobileNetV2 model architecture
model = models.mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 4)
model.load_state_dict(torch.load("mobilenet_finetuned.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Load validation dataset
val_dir = "waste_dataset/cleaned_split/test"
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# ✅ Visualize predictions with GT
plt.figure(figsize=(16, 12))
for idx, (img, label) in enumerate(val_loader):
    if idx >= 12: break
    img = img.to(device)
    label = label.item()

    with torch.no_grad():
        output = model(img)
        probs = torch.softmax(output, dim=1)
        pred = torch.argmax(probs).item()
        confidence = probs[0][pred].item() * 100

    img_np = img.squeeze().permute(1, 2, 0).cpu().numpy()
    plt.subplot(3, 4, idx + 1)
    plt.imshow(img_np)
    title_color = 'green' if pred == label else 'red'
    plt.title(f"Pred: {class_names[pred]} ({confidence:.1f}%)\nGT: {class_names[label]}", color=title_color)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import os
import shutil
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models import mobilenet_v2
import torch.nn as nn

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']

# Load fine-tuned model
model = mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, len(class_names))
model.load_state_dict(torch.load("mobilenet_finetuned.pth", map_location=device))
model.to(device)
model.eval()

# Validation transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# Load validation set
val_dir = "waste_dataset/cleaned_split/test"
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# Directory for misclassified samples
misclassified_dir = "hard_final_tune"
os.makedirs(misclassified_dir, exist_ok=True)

# Identify and copy misclassified samples
misclassified_samples = []
with torch.no_grad():
    for i, (image, label) in enumerate(val_loader):
        image = image.to(device)
        label = label.to(device)
        output = model(image)
        probs = torch.softmax(output, dim=1)
        pred = torch.argmax(probs, dim=1)
        confidence = probs[0][pred.item()].item() * 100

        if pred.item() != label.item():
            img_path, _ = val_dataset.samples[i]
            filename = os.path.basename(img_path)
            shutil.copy(img_path, os.path.join(misclassified_dir, filename))
            misclassified_samples.append((filename, class_names[label.item()], class_names[pred.item()], confidence))

print(f"✅ Copied {len(misclassified_samples)} misclassified images to: {misclassified_dir}")


In [ ]:
import pandas as pd

# Create and display a DataFrame
df_misclassified = pd.DataFrame(
    misclassified_samples,
    columns=["Filename", "True Label", "Predicted Label", "Confidence"]
)

# Show table
print("📊 Misclassified Samples:")
df_misclassified.head(20)  # Show first 20


In [ ]:
import pandas as pd

# Assuming 'moved_files' list is already created
df_moved = pd.DataFrame(moved_files, columns=["Filename", "Moved to Class Folder"])

# Show as table in Jupyter
df_moved.head(20)  # You can change the number to see more


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms, datasets

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 8
num_epochs = 5

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Dataset and DataLoader
data_path = "hard_final_tune"
dataset = datasets.ImageFolder(data_path, transform=transform)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# ✅ Load original model weights
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)

# ✅ Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ✅ Training loop
model.train()
for epoch in range(num_epochs):
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

    acc = 100 * correct / total
    print(f"📘 Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}, Accuracy: {acc:.2f}%")

# ✅ Save fine-tuned model
torch.save(model.state_dict(), "mobilenet_final_tuned.pth")
print("✅ Model saved as mobilenet_final_tuned.pth")


In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os, random
import matplotlib.pyplot as plt

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "waste_dataset/cleaned_split/test"  # Change this if needed
num_images = 5  # number of random images to show

# ✅ Load model
from torchvision import models
import torch.nn as nn
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_finetuned.pth", map_location=device))
model.to(device).eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Collect image paths from all class subfolders
all_image_paths = []
for class_name in class_names:
    folder = os.path.join(test_folder, class_name)
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    all_image_paths.extend(files)

# ✅ Randomly sample images
sampled = random.sample(all_image_paths, num_images)

# ✅ Predict and display
plt.figure(figsize=(num_images * 4, 4))
for i, img_path in enumerate(sampled):
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax().item()
        confidence = probs[0, pred_idx].item() * 100
        pred_label = class_names[pred_idx]

    plt.subplot(1, num_images, i+1)
    plt.imshow(image)
    plt.title(f"{pred_label}\n({confidence:.2f}%)", color="green")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import torch
from torchvision import transforms
from PIL import Image
import os, random
import matplotlib.pyplot as plt

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "waste_dataset/cleaned_split/test"  # Change this if needed
num_images = 5  # number of random images to show

# ✅ Load model
from torchvision import models
import torch.nn as nn
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_finetuned.pth", map_location=device))
model.to(device).eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Collect image paths from all class subfolders
all_image_paths = []
for class_name in class_names:
    folder = os.path.join(test_folder, class_name)
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    all_image_paths.extend(files)

# ✅ Randomly sample images
sampled = random.sample(all_image_paths, num_images)

# ✅ Predict and display
plt.figure(figsize=(num_images * 4, 4))
for i, img_path in enumerate(sampled):
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax().item()
        confidence = probs[0, pred_idx].item() * 100
        pred_label = class_names[pred_idx]

    plt.subplot(1, num_images, i+1)
    plt.imshow(image)
    plt.title(f"{pred_label}\n({confidence:.2f}%)", color="green")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os, random
import matplotlib.pyplot as plt

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "waste_dataset/cleaned_split/test"  # Change this if needed
num_images = 5  # number of random images to show

# ✅ Load model
from torchvision import models
import torch.nn as nn
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_finetuned.pth", map_location=device))
model.to(device).eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Collect image paths from all class subfolders
all_image_paths = []
for class_name in class_names:
    folder = os.path.join(test_folder, class_name)
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    all_image_paths.extend(files)

# ✅ Randomly sample images
sampled = random.sample(all_image_paths, num_images)

# ✅ Predict and display
plt.figure(figsize=(num_images * 4, 4))
for i, img_path in enumerate(sampled):
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax().item()
        confidence = probs[0, pred_idx].item() * 100
        pred_label = class_names[pred_idx]

    plt.subplot(1, num_images, i+1)
    plt.imshow(image)
    plt.title(f"{pred_label}\n({confidence:.2f}%)", color="green")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os, random
import matplotlib.pyplot as plt

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "waste_dataset/cleaned_split/test"  # Change this if needed
num_images = 5  # number of random images to show

# ✅ Load model
from torchvision import models
import torch.nn as nn
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_finetuned.pth", map_location=device))
model.to(device).eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Collect image paths from all class subfolders
all_image_paths = []
for class_name in class_names:
    folder = os.path.join(test_folder, class_name)
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    all_image_paths.extend(files)

# ✅ Randomly sample images
sampled = random.sample(all_image_paths, num_images)

# ✅ Predict and display
plt.figure(figsize=(num_images * 4, 4))
for i, img_path in enumerate(sampled):
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax().item()
        confidence = probs[0, pred_idx].item() * 100
        pred_label = class_names[pred_idx]

    plt.subplot(1, num_images, i+1)
    plt.imshow(image)
    plt.title(f"{pred_label}\n({confidence:.2f}%)", color="green")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os, random
import matplotlib.pyplot as plt

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "waste_dataset/cleaned_split/test"  # Change this if needed
num_images = 5  # number of random images to show

# ✅ Load model
from torchvision import models
import torch.nn as nn
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_finetuned.pth", map_location=device))
model.to(device).eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Collect image paths from all class subfolders
all_image_paths = []
for class_name in class_names:
    folder = os.path.join(test_folder, class_name)
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    all_image_paths.extend(files)

# ✅ Randomly sample images
sampled = random.sample(all_image_paths, num_images)

# ✅ Predict and display
plt.figure(figsize=(num_images * 4, 4))
for i, img_path in enumerate(sampled):
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax().item()
        confidence = probs[0, pred_idx].item() * 100
        pred_label = class_names[pred_idx]

    plt.subplot(1, num_images, i+1)
    plt.imshow(image)
    plt.title(f"{pred_label}\n({confidence:.2f}%)", color="green")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os, random
import matplotlib.pyplot as plt

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "waste_dataset/cleaned_split/test"  # Change this if needed
num_images = 5  # number of random images to show

# ✅ Load model
from torchvision import models
import torch.nn as nn
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_finetuned.pth", map_location=device))
model.to(device).eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Collect image paths from all class subfolders
all_image_paths = []
for class_name in class_names:
    folder = os.path.join(test_folder, class_name)
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    all_image_paths.extend(files)

# ✅ Randomly sample images
sampled = random.sample(all_image_paths, num_images)

# ✅ Predict and display
plt.figure(figsize=(num_images * 4, 4))
for i, img_path in enumerate(sampled):
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax().item()
        confidence = probs[0, pred_idx].item() * 100
        pred_label = class_names[pred_idx]

    plt.subplot(1, num_images, i+1)
    plt.imshow(image)
    plt.title(f"{pred_label}\n({confidence:.2f}%)", color="green")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import os, random
import matplotlib.pyplot as plt

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "waste_dataset/cleaned_split/test"  # Change this if needed
num_images = 5  # number of random images to show

# ✅ Load model
from torchvision import models
import torch.nn as nn
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_finetuned.pth", map_location=device))
model.to(device).eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Collect image paths from all class subfolders
all_image_paths = []
for class_name in class_names:
    folder = os.path.join(test_folder, class_name)
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
    all_image_paths.extend(files)

# ✅ Randomly sample images
sampled = random.sample(all_image_paths, num_images)

# ✅ Predict and display
plt.figure(figsize=(num_images * 4, 4))
for i, img_path in enumerate(sampled):
    image = Image.open(img_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred_idx = probs.argmax().item()
        confidence = probs[0, pred_idx].item() * 100
        pred_label = class_names[pred_idx]

    plt.subplot(1, num_images, i+1)
    plt.imshow(image)
    plt.title(f"{pred_label}\n({confidence:.2f}%)", color="green")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

# ✅ Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
test_folder = "hard_samples"

# ✅ Rebuild the model
model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model = model.to(device)
model.eval()

# ✅ Preprocessing functions
def gray(img): return cv2.cvtColor(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
def blur(img): return cv2.GaussianBlur(np.array(img), (5, 5), 0)
def edges(img): return cv2.cvtColor(cv2.Canny(np.array(img), 100, 200), cv2.COLOR_GRAY2RGB)

techniques = {
    "Original": lambda img: np.array(img),
    "Gray": gray,
    "Blur": blur,
    "Canny": edges
}

# ✅ Transform (same for all)
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Load and process images
image_files = [f for f in os.listdir(test_folder) if f.lower().endswith((".jpg", ".png", ".jpeg"))][:3]

# ✅ Plotting
for fname in image_files:
    fig, axs = plt.subplots(1, len(techniques), figsize=(16, 4))
    path = os.path.join(test_folder, fname)
    img = Image.open(path).convert("RGB")
    fig.suptitle(f"{fname}", fontsize=14)
    
    for i, (tech_name, preprocess) in enumerate(techniques.items()):
        proc_img = preprocess(img)
        input_tensor = transform(proc_img).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            prob = torch.softmax(output, dim=1)
            pred = class_names[prob.argmax().item()]
            confidence = prob.max().item() * 100

        axs[i].imshow(proc_img)
        axs[i].set_title(f"{tech_name}\nPred: {pred}\nConf: {confidence:.1f}%", fontsize=10)
        axs[i].axis("off")
    
    plt.tight_layout()
    plt.show()


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import random
import cv2

# ✅ Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Validation path (change to 'valid' if you prefer)
val_dir = "waste_dataset/cleaned_split/test"

# ✅ Get image paths from subfolders
image_paths = []
for label in class_names:
    class_dir = os.path.join(val_dir, label)
    if os.path.exists(class_dir):
        image_paths.extend([os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])

# ✅ Pick 3 random images
sampled_images = random.sample(image_paths, 3)

# ✅ Load model
model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model = model.to(device)
model.eval()

# ✅ Define preprocessing functions
def gray(img): return cv2.cvtColor(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
def blur(img): return cv2.GaussianBlur(np.array(img), (5, 5), 0)
def edges(img): return cv2.cvtColor(cv2.Canny(np.array(img), 100, 200), cv2.COLOR_GRAY2RGB)

techniques = {
    "Original": lambda img: np.array(img),
    "Gray": gray,
    "Blur": blur,
    "Canny": edges
}

# ✅ Transform
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Prediction and visualization
for path in sampled_images:
    fig, axs = plt.subplots(1, len(techniques), figsize=(16, 4))
    img = Image.open(path).convert("RGB")
    true_label = path.split(os.sep)[-2]
    fig.suptitle(f"{os.path.basename(path)} | GT: {true_label}", fontsize=14)

    for i, (name, preprocess) in enumerate(techniques.items()):
        proc_img = preprocess(img)
        input_tensor = transform(proc_img).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            probs = torch.softmax(output, dim=1)
            pred = class_names[probs.argmax().item()]
            conf = probs.max().item() * 100

        axs[i].imshow(proc_img)
        axs[i].set_title(f"{name}\nPred: {pred}\nConf: {conf:.1f}%", fontsize=10)
        axs[i].axis("off")
    
    plt.tight_layout()
    plt.show()


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt
import random
import cv2

# ✅ Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Validation path (change to 'valid' if you prefer)
val_dir = "waste_dataset/cleaned_split/test"

# ✅ Get image paths from subfolders
image_paths = []
for label in class_names:
    class_dir = os.path.join(val_dir, label)
    if os.path.exists(class_dir):
        image_paths.extend([os.path.join(class_dir, f) for f in os.listdir(class_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))])

# ✅ Pick 3 random images
sampled_images = random.sample(image_paths, 3)

# ✅ Load model
model = models.mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model = model.to(device)
model.eval()

# ✅ Define preprocessing functions
def gray(img): return cv2.cvtColor(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
def blur(img): return cv2.GaussianBlur(np.array(img), (5, 5), 0)
def edges(img): return cv2.cvtColor(cv2.Canny(np.array(img), 100, 200), cv2.COLOR_GRAY2RGB)

techniques = {
    "Original": lambda img: np.array(img),
    "Gray": gray,
    "Blur": blur,
    "Canny": edges
}

# ✅ Transform
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Prediction and visualization
for path in sampled_images:
    fig, axs = plt.subplots(1, len(techniques), figsize=(16, 4))
    img = Image.open(path).convert("RGB")
    true_label = path.split(os.sep)[-2]
    fig.suptitle(f"{os.path.basename(path)} | GT: {true_label}", fontsize=14)

    for i, (name, preprocess) in enumerate(techniques.items()):
        proc_img = preprocess(img)
        input_tensor = transform(proc_img).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            probs = torch.softmax(output, dim=1)
            pred = class_names[probs.argmax().item()]
            conf = probs.max().item() * 100

        axs[i].imshow(proc_img)
        axs[i].set_title(f"{name}\nPred: {pred}\nConf: {conf:.1f}%", fontsize=10)
        axs[i].axis("off")
    
    plt.tight_layout()
    plt.show()


In [ ]:
import os
import torch
import numpy as np
import cv2
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

# ✅ Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load pretrained MobileNetV2 model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Class names
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ✅ Preprocessing methods
def apply_clahe(img):
    lab = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def apply_sharpen(img):
    kernel = np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]])
    return cv2.filter2D(np.array(img), -1, kernel)

def apply_hist_eq(img):
    img_yuv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2YUV)
    img_yuv[:,:,0] = cv2.equalizeHist(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)

def apply_blur(img):
    return cv2.GaussianBlur(np.array(img), (5, 5), 0)

# ✅ Collect all test images
image_dir = "waste_dataset/cleaned_split/test"
all_images = []
for cls in os.listdir(image_dir):
    cls_path = os.path.join(image_dir, cls)
    if os.path.isdir(cls_path):
        for fname in os.listdir(cls_path):
            all_images.append((os.path.join(cls_path, fname), cls))

# ✅ Accuracy tracking
methods = ['CLAHE', 'Sharpen', 'HistEq', 'Blur']
correct = {m: 0 for m in methods}
total = len(all_images)

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(tensor)
        probs = F.softmax(out, dim=1)
        pred = probs.argmax(dim=1).item()
    return class_names[pred]

# ✅ Run predictions
for path, true_label in all_images:
    img = Image.open(path).convert("RGB")
    versions = {
        'CLAHE': apply_clahe(img),
        'Sharpen': apply_sharpen(img),
        'HistEq': apply_hist_eq(img),
        'Blur': apply_blur(img)
    }
    for method, proc_img in versions.items():
        pred_label = predict(proc_img)
        if pred_label == true_label:
            correct[method] += 1

# ✅ Display accuracy
for method in methods:
    acc = 100 * correct[method] / total
    print(f"✅ {method} Accuracy: {acc:.2f}% ({correct[method]}/{total})")


In [ ]:
import os
import torch
import numpy as np
import cv2
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F

# ✅ Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Class names
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Transform
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ✅ Preprocessing methods
def gray(img): return cv2.cvtColor(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
def blur(img): return cv2.GaussianBlur(np.array(img), (5, 5), 0)
def canny(img): return cv2.cvtColor(cv2.Canny(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY), 100, 200), cv2.COLOR_GRAY2RGB)

# ✅ Get all test images
image_folder = "waste_dataset/cleaned_split/test"
all_images = []
for class_name in os.listdir(image_folder):
    class_path = os.path.join(image_folder, class_name)
    if os.path.isdir(class_path):
        for img_name in os.listdir(class_path):
            all_images.append((os.path.join(class_path, img_name), class_name))

# ✅ Accuracy counters
methods = ['Original', 'Gray', 'Blur', 'Canny']
correct_counts = {m: 0 for m in methods}
total = len(all_images)

# ✅ Prediction function
def predict(img_np):
    img_tensor = base_transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img_tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Run predictions
for img_path, true_label in all_images:
    original = Image.open(img_path).convert("RGB")

    imgs = {
        "Original": np.array(original),
        "Gray": gray(original),
        "Blur": blur(original),
        "Canny": canny(original)
    }

    for method, img_np in imgs.items():
        pred_label, _ = predict(img_np)
        if pred_label == true_label:
            correct_counts[method] += 1

# ✅ Print accuracy summary
for method in methods:
    acc = 100 * correct_counts[method] / total
    print(f"{method} Accuracy: {acc:.2f}%")


In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
from torchvision import transforms, models

# ✅ Device and model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.eval().to(device)

# ✅ Preprocessing functions
def gray(img): return cv2.cvtColor(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
def blur(img): return cv2.GaussianBlur(np.array(img), (5, 5), 0)
def canny(img): 
    edges = cv2.Canny(cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY), 100, 200)
    return cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)

# ✅ Transform
transform_base = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Prediction function
def predict(img_np):
    img_tensor = transform_base(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(img_tensor)
        _, predicted = output.max(1)
        confidence = torch.softmax(output, dim=1).max().item()
    return predicted.item(), confidence

# ✅ Dataset folders
dataset_dirs = {
    "Cleaned Waste Dataset": "waste_dataset/cleaned_split/test",
    "Train Data": "train_data/train",
    "TrashNet Dataset": "trashnet_data/dataset-resized"
}
class_list = ["glass", "metal", "paper", "plastic"]
methods = ["Original", "Gray", "Blur", "Canny"]

# ✅ Loop over all datasets
for dataset_name, base_dir in dataset_dirs.items():
    print(f"\n📁 Testing on {dataset_name}...")

    correct_counts = {m: 0 for m in methods}
    total = 0

    for cls in os.listdir(base_dir):
        cls_path = os.path.join(base_dir, cls)
        if not os.path.isdir(cls_path) or cls not in class_list:
            continue
        for fname in os.listdir(cls_path):
            if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
                continue
            img_path = os.path.join(cls_path, fname)
            try:
                original = Image.open(img_path).convert("RGB")
            except:
                continue
            
            imgs = {
                "Original": np.array(original),
                "Gray": gray(original),
                "Blur": blur(original),
                "Canny": canny(original)
            }

            for method, img_np in imgs.items():
                pred_label, _ = predict(img_np)
                if pred_label == class_list.index(cls):
                    correct_counts[method] += 1
            total += 1

    # ✅ Show results
    print("📊 Accuracy for each preprocessing method:")
    for method in methods:
        acc = 100 * correct_counts[method] / total if total > 0 else 0
        print(f"{method:8}: {acc:.2f}%")
    print(f"Total Samples: {total}")


In [ ]:
# ✅ Code to calculate average accuracy

import pandas as pd
import matplotlib.pyplot as plt

# Accuracy values from your 3 datasets
results = {
    "Dataset": ["Cleaned Waste", "Train Data", "TrashNet"],
    "Original": [93.48, 95.91, 95.52],
    "Gray":     [81.70, 86.08, 85.10],
    "Blur":     [92.73, 95.66, 94.36],
    "Canny":    [24.56, 25.10, 24.56]
}

# Convert to DataFrame
df = pd.DataFrame(results)

# Calculate average accuracy for each preprocessing method
averages = df[["Original", "Gray", "Blur", "Canny"]].mean().round(2)

# Display results as a new DataFrame
avg_df = pd.DataFrame({
    "Preprocessing": averages.index,
    "Average Accuracy (%)": averages.values
})

# Show the table
print("📊 Average Accuracy per Preprocessing Method:")
print(avg_df)

# Optional: Visualize as bar chart
plt.figure(figsize=(8, 5))
plt.bar(avg_df["Preprocessing"], avg_df["Average Accuracy (%)"], color='skyblue')
plt.title("Average Accuracy by Preprocessing Method")
plt.ylabel("Accuracy (%)")
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


In [ ]:
import os
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Load model
from torchvision.models import mobilenet_v2
model = mobilenet_v2()
model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model = model.to(device)
model.eval()

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ List of test folders
test_dirs = [
    "waste_dataset/cleaned_split/test",
    "trashnet_data/dataset-resized",
    "train_data/train"  # No test set, so fallback to this
]

y_true = []
y_pred = []

# ✅ Loop through all test folders
for base_dir in test_dirs:
    print(f"\n🔍 Testing on: {base_dir}")
    for cls in os.listdir(base_dir):
        cls_path = os.path.join(base_dir, cls)
        if not os.path.isdir(cls_path) or cls not in class_names:
            continue
        for fname in os.listdir(cls_path):
            img_path = os.path.join(cls_path, fname)
            try:
                from PIL import Image
                image = Image.open(img_path).convert("RGB")
                input_tensor = transform(image).unsqueeze(0).to(device)
                with torch.no_grad():
                    output = model(input_tensor)
                    _, predicted = output.max(1)
                    y_true.append(class_names.index(cls))
                    y_pred.append(predicted.item())
            except:
                print(f"⚠️ Skipped: {img_path}")

# ✅ Print classification report
print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ✅ Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True Label")
plt.title("🧩 Confusion Matrix for All Test Sets")
plt.show()


In [ ]:

import os

# ✅ List of main dataset folders to inspect
folders_to_check = [
    "train_data",
    "waste_dataset",
    "waste_dataset/cleaned_split",
    "trashnet_data"
]

# ✅ Loop through each folder and print contents
for folder in folders_to_check:
    print(f"\n📁 Contents of '{folder}':")
    if os.path.exists(folder):
        for item in os.listdir(folder):
            path = os.path.join(folder, item)
            if os.path.isdir(path):
                print(f"📂 {item}/")
            else:
                print(f"📄 {item}")
    else:
        print("❌ Folder not found.")

In [ ]:
import os

folders = [
    "train_data",
    "waste_dataset",
    "waste_dataset/cleaned_split",
    "trashnet_data"
]

for folder in folders:
    print(f"\n📁 Contents of '{folder}':")
    if os.path.exists(folder):
        print(os.listdir(folder))
    else:
        print("❌ Folder not found.")


In [ ]:
from sklearn.metrics import classification_report

# ✅ تعريف الفئات يدويًا كأرقام حتى لو ما كانت كلها موجودة فعليًا
labels = [0, 1, 2, 3]  # glass=0, metal=1, paper=2, plastic=3

# ✅ طباعة التقارير لكل طريقة معالجة
for method in methods:
    print(f"\n📊 Classification Report for {method}:")
    print(classification_report(
        y_true_all[method],
        y_pred_all[method],
        labels=labels,
        target_names=class_list,
        digits=2,
        zero_division=0  # لتجنب التحذيرات
    ))


In [ ]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Folder containing your hard samples
folder = "hard_samples"
image_files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# Preprocessing functions
def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def apply_sharpen(img):
    blur = cv2.GaussianBlur(img, (0, 0), 3)
    return cv2.addWeighted(img, 1.5, blur, -0.5, 0)

def apply_hist_eq(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def apply_blur(img):
    return cv2.GaussianBlur(img, (5, 5), 0)

# Process each image
for file in image_files:
    img_path = os.path.join(folder, file)
    bgr = cv2.imread(img_path)
    if bgr is None:
        print(f"⚠️ Couldn't read image: {file}")
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Apply all preprocessing
    clahe = apply_clahe(rgb)
    sharpened = apply_sharpen(rgb)
    hist_eq = apply_hist_eq(rgb)
    blurred = apply_blur(rgb)

    # Display results
    plt.figure(figsize=(20, 4))
    plt.suptitle(f"Preprocessing Results - {file}", fontsize=16)
    
    titles = ["Original", "CLAHE", "Sharpened", "Histogram Equalization", "Gaussian Blur"]
    images = [rgb, clahe, sharpened, hist_eq, blurred]

    for i in range(len(images)):
        plt.subplot(1, 5, i+1)
        plt.imshow(images[i])
        plt.title(titles[i])
        plt.axis("off")
    plt.show()


In [ ]:
import os
import torch
import numpy as np
import cv2
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

# ✅ Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load pretrained MobileNetV2 model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Class names
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ✅ Preprocessing methods
def apply_clahe(img):
    lab = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def apply_sharpen(img):
    kernel = np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]])
    return cv2.filter2D(np.array(img), -1, kernel)

def apply_hist_eq(img):
    img_yuv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2YUV)
    img_yuv[:,:,0] = cv2.equalizeHist(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)

def apply_blur(img):
    return cv2.GaussianBlur(np.array(img), (5, 5), 0)

# ✅ Collect all test images
image_dir = "waste_dataset/cleaned_split/test"
all_images = []
for cls in os.listdir(image_dir):
    cls_path = os.path.join(image_dir, cls)
    if os.path.isdir(cls_path):
        for fname in os.listdir(cls_path):
            all_images.append((os.path.join(cls_path, fname), cls))

# ✅ Accuracy tracking
methods = ['CLAHE', 'Sharpen', 'HistEq', 'Blur']
correct = {m: 0 for m in methods}
total = len(all_images)

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(tensor)
        probs = F.softmax(out, dim=1)
        pred = probs.argmax(dim=1).item()
    return class_names[pred]

# ✅ Run predictions
for path, true_label in all_images:
    img = Image.open(path).convert("RGB")
    versions = {
        'CLAHE': apply_clahe(img),
        'Sharpen': apply_sharpen(img),
        'HistEq': apply_hist_eq(img),
        'Blur': apply_blur(img)
    }
    for method, proc_img in versions.items():
        pred_label = predict(proc_img)
        if pred_label == true_label:
            correct[method] += 1

# ✅ Display accuracy
for method in methods:
    acc = 100 * correct[method] / total
    print(f"✅ {method} Accuracy: {acc:.2f}% ({correct[method]}/{total})")


In [ ]:
import os
import torch
import numpy as np
import cv2
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# ✅ Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load Model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Class names
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ✅ Preprocessing functions
def apply_clahe(img):
    lab = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(l)
    return cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2RGB)

def apply_sharpen(img):
    kernel = np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]])
    return cv2.filter2D(np.array(img), -1, kernel)

def apply_hist_eq(img):
    yuv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2YUV)
    yuv[:,:,0] = cv2.equalizeHist(yuv[:,:,0])
    return cv2.cvtColor(yuv, cv2.COLOR_YUV2RGB)

def apply_blur(img):
    return cv2.GaussianBlur(np.array(img), (5, 5), 0)

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(tensor)
        probs = F.softmax(out, dim=1)
        pred = probs.argmax(dim=1).item()
    return pred

# ✅ Load hard sample filenames
image_dir = "hard_samples"
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

def get_true_label(fname):
    label = fname.split("_")[0].lower()
    return class_names.index(label) if label in class_names else -1

# ✅ Preprocessing methods
methods = {
    "Original": lambda img: np.array(img),
    "CLAHE": apply_clahe,
    "Sharpen": apply_sharpen,
    "HistEq": apply_hist_eq,
    "Blur": apply_blur,
}

results = {}
for method_name, method_func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        img_path = os.path.join(image_dir, fname)
        true_label = get_true_label(fname)
        if true_label == -1:
            continue
        img = Image.open(img_path).convert("RGB")
        proc_img = method_func(img)
        pred_label = predict(proc_img)
        y_true.append(true_label)
        y_pred.append(pred_label)
    results[method_name] = {
        "y_true": y_true,
        "y_pred": y_pred,
        "report": classification_report(
            y_true, y_pred,
            labels=[0, 1, 2, 3],
            target_names=class_names,
            output_dict=True,
            zero_division=0
        ),
        "conf_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    }

# ✅ Print summary
summary_df = pd.DataFrame([
    {
        "Method": method,
        "Accuracy": result["report"]["accuracy"],
        "Precision (macro avg)": result["report"]["macro avg"]["precision"],
        "Recall (macro avg)": result["report"]["macro avg"]["recall"],
        "F1-score (macro avg)": result["report"]["macro avg"]["f1-score"]
    }
    for method, result in results.items()
])
print("📊 Classification Summary:")
print(summary_df)


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
from sklearn.metrics import classification_report, accuracy_score

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing functions
def apply_clahe(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(img_np, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    result = cv2.cvtColor(cv2.cvtColor(merged, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)
    return result

def apply_sharpen(img):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), 1.0)
    sharpened = cv2.addWeighted(img_np, 1.5, blurred, -0.5, 0)
    return sharpened

def apply_histeq(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(img_np)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def apply_blur(img):
    return cv2.GaussianBlur(np.array(img), (5, 5), 0)

def apply_clahe_sharpen(img):
    clahe_img = apply_clahe(img)
    sharpened = apply_sharpen(Image.fromarray(clahe_img))
    return sharpened

# ✅ Prediction function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Run for all preprocessing methods
methods = {
    "Original": lambda x: np.array(x),
    "CLAHE": apply_clahe,
    "Sharpen": apply_sharpen,
    "HistEq": apply_histeq,
    "Blur": apply_blur,
    "CLAHE+Sharpen": apply_clahe_sharpen
}

results = {}

image_files = [f for f in os.listdir(hard_dir) if f.endswith((".jpg", ".png", ".jpeg"))]

for method_name, func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        path = os.path.join(hard_dir, fname)
        true_label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not true_label:
            continue
        img = Image.open(path).convert("RGB")
        proc_img = func(img)
        pred_label, _ = predict(proc_img)
        y_true.append(true_label)
        y_pred.append(pred_label)

    report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, zero_division=0, output_dict=True)
    results[method_name] = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": report["macro avg"]["precision"],
        "recall": report["macro avg"]["recall"],
        "f1": report["macro avg"]["f1-score"]
    }

# ✅ Display Summary
import pandas as pd
df = pd.DataFrame([
    {"Method": k,
     "Accuracy": v["accuracy"] * 100,
     "Precision": v["precision"] * 100,
     "Recall": v["recall"] * 100,
     "F1-score": v["f1"] * 100}
    for k, v in results.items()
])
df = df.sort_values(by="F1-score", ascending=False).reset_index(drop=True)
print("\n📊 Performance Comparison on Hard Samples:")
print(df.round(2))


In [ ]:
import os

# List images in hard_samples
image_dir = "hard_samples"
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
print("Available sample images in 'hard_samples':")
for f in image_files:
    print(f)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image path
sample_name = "glass_3.jpg"  # Change to any name from your hard_samples list
sample_path = f"hard_samples/{sample_name}"
img = cv2.imread(sample_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# ✅ Preprocessing functions
def to_tensor(img): return transforms.ToTensor()(Image.fromarray(img)).unsqueeze(0).to(device)

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def histogram_eq(img):
    img_yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
    img_yuv[:, :, 0] = cv2.equalizeHist(img_yuv[:, :, 0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)

def sharpen(img):
    kernel = np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)

def clahe_sharpen(img):
    return sharpen(apply_clahe(img))

# ✅ Predict function
def predict(img):
    with torch.no_grad():
        out = model(to_tensor(img))
        probs = F.softmax(out, dim=1)
        conf, pred = probs.max(1)
        return class_names[pred.item()], conf.item()

# ✅ Test and plot
methods = {
    "Original": img_rgb,
    "CLAHE": apply_clahe(img_rgb),
    "HistEq": histogram_eq(img_rgb),
    "Sharpen": sharpen(img_rgb),
    "CLAHE+Sharpen": clahe_sharpen(img_rgb)
}

plt.figure(figsize=(15, 4))
for i, (name, processed) in enumerate(methods.items()):
    label, conf = predict(processed)
    plt.subplot(1, 5, i+1)
    plt.imshow(processed)
    plt.title(f"{name}\n{label} ({conf*100:.1f}%)")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# ✅ Device and setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Base transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing functions
def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=2.0).apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def histogram_eq(img):
    yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
    yuv[:, :, 0] = cv2.equalizeHist(yuv[:, :, 0])
    return cv2.cvtColor(yuv, cv2.COLOR_YUV2RGB)

def sharpen(img):
    kernel = np.array([[0, -1, 0], [-1, 5,-1], [0, -1, 0]])
    return cv2.filter2D(img, -1, kernel)

def clahe_sharpen(img):
    return sharpen(apply_clahe(img))

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(tensor)
        probs = F.softmax(out, dim=1)
        conf, pred = probs.max(1)
        return class_names[pred.item()], conf.item()

# ✅ Load hard samples
hard_folder = "hard_samples"
image_files = [f for f in os.listdir(hard_folder) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

# ✅ Prepare tracking
methods = {
    "Original": lambda x: x,
    "CLAHE": apply_clahe,
    "HistEq": histogram_eq,
    "Sharpen": sharpen,
    "CLAHE+Sharpen": clahe_sharpen
}
results = {}

for method_name, func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        img_path = os.path.join(hard_folder, fname)
        img = Image.open(img_path).convert("RGB")
        img_np = np.array(img)
        processed = func(img_np)

        pred_label, _ = predict(processed)

        # infer true label from filename
        true_label = None
        for cname in class_names:
            if cname in fname.lower():
                true_label = cname
                break
        if true_label:
            y_true.append(true_label)
            y_pred.append(pred_label)

    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
    acc = 100 * sum([yt == yp for yt, yp in zip(y_true, y_pred)]) / len(y_true)
    results[method_name] = {
        "Accuracy": acc,
        "Precision": report["macro avg"]["precision"] * 100,
        "Recall": report["macro avg"]["recall"] * 100,
        "F1-score": report["macro avg"]["f1-score"] * 100
    }

# ✅ Show summary
df_summary = pd.DataFrame([
    {"Method": k, **v} for k, v in results.items()
])
print("📊 Performance Comparison on Hard Samples:")
print(df_summary)


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing functions
def apply_clahe(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(img_np, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    result = cv2.cvtColor(cv2.cvtColor(merged, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)
    return result

def apply_sharpen(img):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), 1.0)
    sharpened = cv2.addWeighted(img_np, 1.5, blurred, -0.5, 0)
    return sharpened

def apply_histeq(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(img_np)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def apply_bilateral(img):
    img_np = np.array(img)
    filtered = cv2.bilateralFilter(img_np, d=9, sigmaColor=75, sigmaSpace=75)
    return filtered

def apply_clahe_sharpen(img):
    clahe_img = apply_clahe(img)
    sharpened = apply_sharpen(Image.fromarray(clahe_img))
    return sharpened

# ✅ Prediction function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Run for all preprocessing methods
methods = {
    "Original": lambda x: np.array(x),
    "CLAHE": apply_clahe,
    "Sharpen": apply_sharpen,
    "HistEq": apply_histeq,
    "Bilateral": apply_bilateral,
    "CLAHE+Sharpen": apply_clahe_sharpen
}

results = {}

image_files = [f for f in os.listdir(hard_dir) if f.endswith((".jpg", ".png", ".jpeg"))]

for method_name, func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        path = os.path.join(hard_dir, fname)
        true_label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not true_label:
            continue
        img = Image.open(path).convert("RGB")
        proc_img = func(img)
        pred_label, _ = predict(proc_img)
        y_true.append(true_label)
        y_pred.append(pred_label)

    report = classification_report(
        y_true, y_pred,
        labels=class_names,
        target_names=class_names,
        zero_division=0,
        output_dict=True
    )
    results[method_name] = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": report["macro avg"]["precision"],
        "recall": report["macro avg"]["recall"],
        "f1": report["macro avg"]["f1-score"]
    }

# ✅ Display Summary
df = pd.DataFrame([
    {"Method": k,
     "Accuracy": v["accuracy"] * 100,
     "Precision": v["precision"] * 100,
     "Recall": v["recall"] * 100,
     "F1-score": v["f1"] * 100}
    for k, v in results.items()
])
df = df.sort_values(by="F1-score", ascending=False).reset_index(drop=True)
print("\n📊 Performance Comparison on Hard Samples:")
print(df.round(2))


In [ ]:
import os
from PIL import Image
import torch
from torchvision import transforms
import matplotlib.pyplot as plt

# ✅ Folder of images
folder_path = "hard_samples"
image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

# ✅ Transform: Resize and Normalize (ImageNet stats)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],  # ImageNet mean
                         std=[0.229, 0.224, 0.225])   # ImageNet std
])

# ✅ Loop and visualize
for fname in image_files:
    path = os.path.join(folder_path, fname)
    image = Image.open(path).convert("RGB")
    
    # Apply transform
    tensor = transform(image)

    # For visualization (unnormalize)
    unnorm = tensor.clone()
    unnorm[0] = unnorm[0] * 0.229 + 0.485
    unnorm[1] = unnorm[1] * 0.224 + 0.456
    unnorm[2] = unnorm[2] * 0.225 + 0.406
    unnorm_img = transforms.ToPILImage()(unnorm)

    # Show both
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(unnorm_img)
    plt.title("Resized + Normalized")
    plt.axis("off")

    plt.suptitle(fname, fontsize=10)
    plt.tight_layout()
    plt.show()


In [ ]:
# Re-import necessary libraries after kernel reset
import os
import numpy as np
import torch
import cv2
import pandas as pd
from PIL import Image
from sklearn.metrics import classification_report, accuracy_score
from torchvision import transforms
from torchvision.models import mobilenet_v2

# ✅ Device & Model Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Functions
def hist_eq(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def sharpen(img):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), 1.0)
    sharp = cv2.addWeighted(img_np, 1.5, blurred, -0.5, 0)
    return sharp

def clahe(img):
    lab = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    c = cv2.createCLAHE(clipLimit=2.0).apply(l)
    merged = cv2.merge((c, a, b))
    return cv2.cvtColor(cv2.cvtColor(merged, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)

def bilateral(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

# ✅ Combined Methods
def hist_eq_sharpen(img): return sharpen(Image.fromarray(hist_eq(img)))
def hist_eq_clahe(img): return clahe(Image.fromarray(hist_eq(img)))
def hist_eq_bilateral(img): return bilateral(Image.fromarray(hist_eq(img)))

# ✅ Predict Function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Dataset & Evaluation
methods = {
    "Original": lambda x: np.array(x),
    "HistEq": hist_eq,
    "HistEq+Sharpen": hist_eq_sharpen,
    "HistEq+CLAHE": hist_eq_clahe,
    "HistEq+Bilateral": hist_eq_bilateral
}

hard_dir = "hard_samples"

image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
results = {}

for name, func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not label:
            continue
        img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
        proc = func(img)
        pred, _ = predict(proc)
        y_true.append(label)
        y_pred.append(pred)

    report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, zero_division=0, output_dict=True)
    results[name] = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": report["macro avg"]["precision"],
        "recall": report["macro avg"]["recall"],
        "f1": report["macro avg"]["f1-score"]
    }

# ✅ Summary DataFrame
df = pd.DataFrame([
    {
        "Method": method,
        "Accuracy": metrics["accuracy"] * 100,
        "Precision": metrics["precision"] * 100,
        "Recall": metrics["recall"] * 100,
        "F1-score": metrics["f1"] * 100
    } for method, metrics in results.items()
])
df = df.sort_values("F1-score", ascending=False).reset_index(drop=True)
print("\n📊 Combined Preprocessing Comparison on Hard Samples:")
print(df.round(2))


In [ ]:
import os

print("📂 Available .pth files in current directory:")
for file in os.listdir():
    if file.endswith(".pth"):
        print("✅", file)


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Base transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing methods
def hist_eq(img):
    img_gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(img_gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def hist_eq_clahe(img):
    img_bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    result = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
    return result

def hist_eq_sharpen(img):
    img_eq = hist_eq(img)
    blurred = cv2.GaussianBlur(img_eq, (0, 0), 1.0)
    sharpened = cv2.addWeighted(img_eq, 1.5, blurred, -0.5, 0)
    return sharpened

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Get list of images (limit to 12 for visualization)
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
image_files = image_files[:12]

# ✅ Display results
n_methods = 4
methods = {
    "Original": lambda x: np.array(x),
    "HistEq": hist_eq,
    "HistEq+CLAHE": hist_eq_clahe,
    "HistEq+Sharpen": hist_eq_sharpen
}

fig, axes = plt.subplots(len(image_files), n_methods, figsize=(n_methods * 4, len(image_files) * 2.5))
for row_idx, fname in enumerate(image_files):
    path = os.path.join(hard_dir, fname)
    img = Image.open(path).convert("RGB")
    for col_idx, (method_name, func) in enumerate(methods.items()):
        processed_img = func(img)
        axes[row_idx, col_idx].imshow(processed_img)
        axes[row_idx, col_idx].axis("off")
        if row_idx == 0:
            axes[row_idx, col_idx].set_title(method_name, fontsize=10)
        if col_idx == 0:
            axes[row_idx, col_idx].set_ylabel(fname, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing
def hist_eq(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def hist_eq_clahe(img):
    img_bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(cv2.cvtColor(merged, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)

# ✅ Predict
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Run and collect results
methods = {
    "Original": lambda x: np.array(x),
    "HistEq": hist_eq,
    "HistEq+CLAHE": hist_eq_clahe
}

image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
results = {}

for name, func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        label = next((c for c in class_names if c in fname.lower()), None)
        if not label: continue
        img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
        proc = func(img)
        pred, _ = predict(proc)
        y_true.append(label)
        y_pred.append(pred)
    report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
    results[name] = {
        "Accuracy": accuracy_score(y_true, y_pred) * 100,
        "Precision": report["macro avg"]["precision"] * 100,
        "Recall": report["macro avg"]["recall"] * 100,
        "F1-score": report["macro avg"]["f1-score"] * 100
    }

# ✅ Display
df = pd.DataFrame([
    {"Method": m, **v} for m, v in results.items()
]).sort_values("F1-score", ascending=False).reset_index(drop=True)
print("\n📊 Combined Preprocessing Comparison on Hard Samples:")
print(df.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing techniques
def hist_eq(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(img_np)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def clahe_eq(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(img_np, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    result = cv2.cvtColor(cv2.cvtColor(merged, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)
    return result

def hist_eq_plus_clahe(img):
    img = Image.fromarray(hist_eq(img))
    return clahe_eq(img)

# ✅ Prediction function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Run evaluation
methods = {
    "Original": lambda x: np.array(x),
    "HistEq": hist_eq,
    "HistEq+CLAHE": hist_eq_plus_clahe
}

results = {}
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

for method_name, func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        path = os.path.join(hard_dir, fname)
        true_label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not true_label:
            continue
        img = Image.open(path).convert("RGB")
        processed = func(img)
        pred_label, _ = predict(processed)
        y_true.append(true_label)
        y_pred.append(pred_label)
    
    report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, zero_division=0, output_dict=True)
    results[method_name] = {
        "accuracy": accuracy_score(y_true, y_pred) * 100,
        "precision": report["macro avg"]["precision"] * 100,
        "recall": report["macro avg"]["recall"] * 100,
        "f1": report["macro avg"]["f1-score"] * 100
    }

# ✅ Display results
df_results = pd.DataFrame([
    {"Method": k, "Accuracy": v["accuracy"], "Precision": v["precision"], "Recall": v["recall"], "F1-score": v["f1"]}
    for k, v in results.items()
]).sort_values(by="F1-score", ascending=False).reset_index(drop=True)

print("\n📊 Combined Preprocessing Comparison on Hard Samples:")
print(df_results.round(2))


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ✅ Fixed MSRCR Function
def msrcr(img, sigma_list=None, G=5, b=25, alpha=125, beta=46):
    if sigma_list is None:
        sigma_list = [15, 80, 250]

    img = img.astype(np.float32) + 1.0
    img_retinex = np.zeros_like(img)

    for sigma in sigma_list:
        blur = cv2.GaussianBlur(img, (0, 0), sigma)
        img_retinex += np.log10(img) - np.log10(blur + 1)

    img_retinex /= len(sigma_list)

    for i in range(img.shape[2]):
        unique, count = np.unique(img_retinex[:, :, i], return_counts=True)
        cumsum = np.cumsum(count)

        N = img.shape[0] * img.shape[1]
        low_idx = np.searchsorted(cumsum, 0.01 * N)
        high_idx = np.searchsorted(cumsum, 0.99 * N)

        low_val = unique[min(len(unique) - 1, low_idx)]
        high_val = unique[min(len(unique) - 1, high_idx)]

        if high_val - low_val > 0:
            img_retinex[:, :, i] = np.clip((img_retinex[:, :, i] - low_val) / (high_val - low_val), 0, 1)
        else:
            img_retinex[:, :, i] = np.zeros_like(img_retinex[:, :, i])

    intensity = img.sum(axis=2) / img.shape[2]
    retinex_intensity = img_retinex.sum(axis=2) / img_retinex.shape[2]

    intensity = G * (np.log10(alpha * intensity + 1) - np.log10(retinex_intensity + 1))
    intensity = np.clip((intensity - intensity.min()) / (intensity.max() - intensity.min()), 0, 1)

    img_msrcr = img_retinex * (intensity[:, :, np.newaxis] + beta)
    img_msrcr = np.clip((img_msrcr - img_msrcr.min()) / (img_msrcr.max() - img_msrcr.min()) * 255, 0, 255).astype(np.uint8)

    return img_msrcr


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ✅ Path to your hard sample images
folder = "hard_samples"
image_files = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

# ✅ MSRCR Enhancement Function
def msrcr(img, sigma_list=[15, 80, 250], G=5, b=25, alpha=125, beta=46):
    img = img.astype(np.float32) + 1.0
    img_retinex = np.zeros_like(img)
    for i in range(3):
        channel = img[:, :, i]
        retinex = np.zeros_like(channel)
        for sigma in sigma_list:
            blur = cv2.GaussianBlur(channel, (0, 0), sigma)
            retinex += np.log10(channel) - np.log10(blur + 1e-6)
        retinex /= len(sigma_list)
        img_retinex[:, :, i] = retinex

    # Color restoration
    intensity = img.sum(axis=2) / 3
    img_color = np.zeros_like(img)
    for i in range(3):
        img_color[:, :, i] = beta * (np.log10(alpha * (img[:, :, i] + 1) / (intensity + 1)))
    
    img_msrcr = G * (img_retinex * img_color + b)
    img_msrcr = (img_msrcr - np.min(img_msrcr)) / (np.max(img_msrcr) - np.min(img_msrcr)) * 255
    return np.uint8(np.clip(img_msrcr, 0, 255))

# ✅ Optional Guided Filter (requires opencv-contrib-python)
def guided_filter(img, radius=5, eps=1e-3):
    import cv2.ximgproc
    guide = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    filtered = cv2.ximgproc.guidedFilter(guide=guide, src=img, radius=radius, eps=eps)
    return filtered

# ✅ Plot results
cols = 2
rows = len(image_files)
plt.figure(figsize=(10, rows * 3))

for i, fname in enumerate(image_files):
    path = os.path.join(folder, fname)
    img = cv2.imread(path)
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    enhanced = msrcr(img_rgb)  # or use: guided_filter(msrcr(img_rgb))

    # Original
    plt.subplot(rows, cols, i * 2 + 1)
    plt.imshow(img_rgb)
    plt.title(f"Original: {fname}")
    plt.axis("off")

    # Enhanced
    plt.subplot(rows, cols, i * 2 + 2)
    plt.imshow(enhanced)
    plt.title("MSRCR Enhanced")
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import os
import cv2
import numpy as np
from PIL import Image
import torch
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
folder = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ MSRCR Enhancement
def msrcr(img, sigma_list=[15, 80, 250], G=5, b=25, alpha=125, beta=46):
    img = img.astype(np.float32) + 1.0
    img_retinex = np.zeros_like(img)
    for i in range(3):
        channel = img[:, :, i]
        retinex = np.zeros_like(channel)
        for sigma in sigma_list:
            blur = cv2.GaussianBlur(channel, (0, 0), sigma)
            retinex += np.log10(channel) - np.log10(blur + 1e-6)
        retinex /= len(sigma_list)
        img_retinex[:, :, i] = retinex

    intensity = img.sum(axis=2) / 3
    img_color = np.zeros_like(img)
    for i in range(3):
        img_color[:, :, i] = beta * (np.log10(alpha * (img[:, :, i] + 1) / (intensity + 1)))

    img_msrcr = G * (img_retinex * img_color + b)
    img_msrcr = (img_msrcr - np.min(img_msrcr)) / (np.max(img_msrcr) - np.min(img_msrcr)) * 255
    return np.uint8(np.clip(img_msrcr, 0, 255))

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Run predictions
y_true = []
y_pred = []

image_files = [f for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
for fname in image_files:
    path = os.path.join(folder, fname)
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not true_label:
        continue

    img = cv2.imread(path)
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    enhanced = msrcr(img_rgb)
    pred_label, _ = predict(enhanced)
    y_true.append(true_label)
    y_pred.append(pred_label)

# ✅ Classification report
report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, zero_division=0, output_dict=True)
accuracy = accuracy_score(y_true, y_pred)

# ✅ Print summary
print("\n📊 Classification Report for MSRCR Enhanced Images:\n")
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
print(f"✅ Accuracy: {accuracy * 100:.2f}%")


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Histogram Equalization + Light Sharpening
def hist_eq_sharpen(img):
    img_np = np.array(img)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    hist_eq = cv2.equalizeHist(gray)
    hist_eq_rgb = cv2.cvtColor(hist_eq, cv2.COLOR_GRAY2RGB)
    blurred = cv2.GaussianBlur(hist_eq_rgb, (0, 0), 1.0)
    sharpened = cv2.addWeighted(hist_eq_rgb, 1.3, blurred, -0.3, 0)
    return sharpened

# ✅ Prediction function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Evaluate
y_true, y_pred = [], []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

for fname in image_files:
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not true_label:
        continue
    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")
    processed_img = hist_eq_sharpen(img)
    pred_label, _ = predict(processed_img)
    y_true.append(true_label)
    y_pred.append(pred_label)

# ✅ Report
print("\n📊 Classification Report for HistEq + Light Sharpening:\n")
report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
print(report)
print(f"✅ Accuracy: {accuracy_score(y_true, y_pred)*100:.2f}%")


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing: HistEq + Bilateral Filter
def hist_eq_bilateral(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    hist_eq = cv2.equalizeHist(gray)
    hist_eq_rgb = cv2.cvtColor(hist_eq, cv2.COLOR_GRAY2RGB)
    bilateral = cv2.bilateralFilter(hist_eq_rgb, d=9, sigmaColor=75, sigmaSpace=75)
    return bilateral

# ✅ Prediction function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Run on all images
y_true, y_pred = [], []
for fname in os.listdir(hard_dir):
    if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    path = os.path.join(hard_dir, fname)
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not true_label:
        continue
    img = Image.open(path).convert("RGB")
    enhanced = hist_eq_bilateral(img)
    pred_label, _ = predict(enhanced)
    y_true.append(true_label)
    y_pred.append(pred_label)

# ✅ Evaluation
report = classification_report(y_true, y_pred, target_names=class_names, digits=2, output_dict=False)
accuracy = accuracy_score(y_true, y_pred)
print("\n📊 Classification Report for HistEq + Bilateral:")
print(report)
print(f"\n✅ Accuracy: {accuracy * 100:.2f}%")


In [ ]:
import os
import cv2
import numpy as np
import torch
from torchvision import transforms
from torchvision.models import mobilenet_v2
from sklearn.metrics import classification_report, accuracy_score
from PIL import Image
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing functions
def hist_eq_unsharp(img):
    img_np = np.array(img)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    eq_rgb = cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)
    blurred = cv2.GaussianBlur(eq_rgb, (0, 0), 1.0)
    sharpened = cv2.addWeighted(eq_rgb, 1.5, blurred, -0.5, 0)
    return sharpened

def adaptive_clahe_lab(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(img_np)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    result = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
    return result

# ✅ Prediction function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()]

# ✅ Run evaluation
methods = {
    "HistEq+Unsharp": hist_eq_unsharp,
    "AdaptiveCLAHE": adaptive_clahe_lab
}

results = {}
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

for name, func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        path = os.path.join(hard_dir, fname)
        true_label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not true_label:
            continue
        img = Image.open(path).convert("RGB")
        proc_img = func(img)
        pred_label = predict(proc_img)
        y_true.append(true_label)
        y_pred.append(pred_label)

    report = classification_report(
        y_true, y_pred, labels=class_names,
        target_names=class_names, output_dict=True, zero_division=0
    )
    results[name] = {
        "accuracy": accuracy_score(y_true, y_pred) * 100,
        "precision": report["macro avg"]["precision"] * 100,
        "recall": report["macro avg"]["recall"] * 100,
        "f1": report["macro avg"]["f1-score"] * 100
    }

# ✅ Display summary
df = pd.DataFrame([
    {"Method": k, "Accuracy": v["accuracy"], "Precision": v["precision"], 
     "Recall": v["recall"], "F1-score": v["f1"]}
    for k, v in results.items()
])
df = df.sort_values("F1-score", ascending=False).reset_index(drop=True)
print("📊 Enhanced Preprocessing Comparison on Hard Samples:\n")
print(df.round(2))


In [ ]:
from torchvision import transforms

# ✅ Data Augmentation for Training
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),            # Random zoom & crop
    transforms.RandomHorizontalFlip(p=0.5),                         # Flip horizontally
    transforms.RandomRotation(degrees=15),                          # Slight rotation
    transforms.ColorJitter(brightness=0.2, contrast=0.2, 
                           saturation=0.2, hue=0.05),               # Adjust brightness/contrast/color
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),  # Random blur
    transforms.ToTensor(),                                          # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406],                     # Normalize using ImageNet stats
                         [0.229, 0.224, 0.225])
])

# ✅ Validation/Test Transform (no augmentation, only resize and normalize)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


In [ ]:
import os
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torchvision

# ✅ Define the path to your dataset
data_dir = "waste_dataset/cleaned_split/train"  # replace with your folder path

# ✅ Define data augmentation transform
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=(3, 3))], p=0.3),
    transforms.ToTensor(),
])

# ✅ Load the dataset with augmentations
train_dataset = datasets.ImageFolder(root=data_dir, transform=train_transform)

# ✅ Create a DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# ✅ Display some augmented images
def imshow(img_tensor):
    img = img_tensor.numpy().transpose((1, 2, 0))
    plt.imshow(img)
    plt.axis("off")

# Get a batch of images
images, labels = next(iter(train_loader))

# Show the images
plt.figure(figsize=(10, 5))
for i in range(4):
    plt.subplot(1, 4, i + 1)
    imshow(images[i])
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torchvision import transforms

# ✅ Find a random image from your dataset
base_folder = "waste_dataset/cleaned_split/test"
classes = os.listdir(base_folder)
random_class = random.choice(classes)
class_path = os.path.join(base_folder, random_class)
image_name = random.choice(os.listdir(class_path))
image_path = os.path.join(class_path, image_name)

print(f"Using image: {image_path}")

# ✅ Define augmentation
augmentation = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15, fill=(255, 255, 255)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.05),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),
    transforms.ToTensor()
])

# ✅ Load and transform
original_img = Image.open(image_path).convert("RGB")
augmented_img = augmentation(original_img)
original_tensor = transforms.ToTensor()(original_img)

# ✅ Display
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original_tensor.permute(1, 2, 0))
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(augmented_img.permute(1, 2, 0))
axes[1].set_title("Augmented Image")
axes[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder

# ✅ Define stronger data augmentation
strong_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.5, 1.0)),         # Stronger zoom
    transforms.RandomHorizontalFlip(p=0.7),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.5, contrast=0.5,
                           saturation=0.5, hue=0.1),
    transforms.RandomGrayscale(p=0.3),
    transforms.RandomPerspective(distortion_scale=0.5, p=0.5),
    transforms.ToTensor()
])

# ✅ Load a random image from your test dataset
dataset_path = "waste_dataset/cleaned_split/test"
classes = os.listdir(dataset_path)
selected_class = random.choice(classes)
class_folder = os.path.join(dataset_path, selected_class)
image_files = [f for f in os.listdir(class_folder) if f.endswith((".jpg", ".png", ".jpeg"))]
image_path = os.path.join(class_folder, random.choice(image_files))

# ✅ Load image
original_image = Image.open(image_path).convert("RGB")
augmented_tensor = strong_transform(original_image)
augmented_image = transforms.ToPILImage()(augmented_tensor)

# ✅ Show side by side
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(original_image)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(augmented_image)
plt.title("Strong Augmentation")
plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F

# ✅ إعداد الجهاز والموديل
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ التحويل الأساسي للصورة
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ دوال المعالجة المسبقة
def apply_clahe(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(img_np, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def apply_sharpen(img):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), 1.0)
    return cv2.addWeighted(img_np, 1.5, blurred, -0.5, 0)

def apply_histeq(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def apply_bilateral(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    filtered = cv2.bilateralFilter(img_np, 9, 75, 75)
    return cv2.cvtColor(filtered, cv2.COLOR_BGR2RGB)

def apply_gamma(img, gamma=1.5):
    invGamma = 1.0 / gamma
    table = np.array([(i / 255.0) ** invGamma * 255 for i in range(256)]).astype("uint8")
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    adjusted = cv2.LUT(img_np, table)
    return cv2.cvtColor(adjusted, cv2.COLOR_BGR2RGB)

# ✅ التنبؤ بالتصنيف
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item() * 100

# ✅ قائمة طرق المعالجة
methods = {
    "Original": lambda x: np.array(x),
    "CLAHE": apply_clahe,
    "Sharpen": apply_sharpen,
    "HistEq": apply_histeq,
    "Bilateral": apply_bilateral,
    "Gamma": apply_gamma
}

# ✅ اختيار 5 صور من hard_samples
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
sample_images = image_files[:5]

# ✅ عرض النتائج
for fname in sample_images:
    img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
    plt.figure(figsize=(16, 5))
    plt.suptitle(f"🖼️ {fname}", fontsize=14)

    for idx, (method, func) in enumerate(methods.items()):
        processed = func(img)
        pred, conf = predict(processed)

        plt.subplot(1, len(methods), idx + 1)
        plt.imshow(processed)
        plt.axis("off")
        plt.title(f"{method}\n🧠 {pred} ({conf:.1f}%)", fontsize=10, color="green")

    plt.tight_layout()
    plt.show()


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ✅ Hard samples directory
hard_samples_dir = "hard_samples"
image_files = [f for f in os.listdir(hard_samples_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
image_paths = [os.path.join(hard_samples_dir, f) for f in image_files]

# ✅ Preprocessing methods
def apply_clahe(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(img_np, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0)
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    result = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)
    return result

def apply_sharpen(img):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), 3)
    sharpened = cv2.addWeighted(img_np, 1.5, blurred, -0.5, 0)
    return sharpened

def apply_histeq(img):
    img_np = np.array(img)
    channels = cv2.split(img_np)
    eq_channels = [cv2.equalizeHist(c) for c in channels]
    eq_img = cv2.merge(eq_channels)
    return eq_img

def apply_bilateral(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

# ✅ Dictionary of methods
methods = {
    "Original": lambda img: np.array(img),
    "CLAHE": apply_clahe,
    "Sharpen": apply_sharpen,
    "HistEq": apply_histeq,
    "Bilateral": apply_bilateral
}

# ✅ Show a few images for comparison
num_to_show = 3
selected_paths = image_paths[:num_to_show]

fig, axs = plt.subplots(len(selected_paths), len(methods), figsize=(15, 5 * len(selected_paths)))

for row, img_path in enumerate(selected_paths):
    img = Image.open(img_path).convert("RGB")
    for col, (method_name, func) in enumerate(methods.items()):
        processed = func(img)
        axs[row, col].imshow(processed)
        axs[row, col].set_title(f"{method_name}", fontsize=12)
        axs[row, col].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import cv2
import torch
from torchvision import transforms
from torchvision.models import mobilenet_v2
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ✅ Preprocessing Methods
def apply_clahe(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(img_np, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=2.0).apply(l)
    lab = cv2.merge((cl, a, b))
    return cv2.cvtColor(cv2.cvtColor(lab, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)

def apply_sharpen(img):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), 1.0)
    return cv2.addWeighted(img_np, 1.5, blurred, -0.5, 0)

def apply_histeq(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(img_np)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def apply_bilateral(img):
    img_np = np.array(img)
    return cv2.bilateralFilter(img_np, 9, 75, 75)

preprocessing_methods = {
    "Original": lambda img: np.array(img),
    "CLAHE": apply_clahe,
    "Sharpen": apply_sharpen,
    "HistEq": apply_histeq,
    "Bilateral": apply_bilateral
}

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Load images and evaluate
results = {k: {"y_true": [], "y_pred": []} for k in preprocessing_methods}

image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

for fname in image_files:
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not true_label:
        continue
    path = os.path.join(hard_dir, fname)
    img = Image.open(path).convert("RGB")

    plt.figure(figsize=(15, 3))
    for i, (name, method) in enumerate(preprocessing_methods.items()):
        proc = method(img)
        pred, conf = predict(proc)
        results[name]["y_true"].append(true_label)
        results[name]["y_pred"].append(pred)

        plt.subplot(1, 5, i + 1)
        plt.imshow(proc)
        plt.axis("off")
        plt.title(f"{name}\nPred: {pred}\nConf: {conf*100:.1f}%", fontsize=9)
    plt.tight_layout()
    plt.show()

# ✅ Accuracy Summary
summary = []
for method, values in results.items():
    acc = accuracy_score(values["y_true"], values["y_pred"]) * 100
    summary.append({"Method": method, "Accuracy": acc})

df_summary = pd.DataFrame(summary).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
print("\n📊 Preprocessing Accuracy Summary:\n")
print(df_summary.round(2))


In [ ]:
#start from here #


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Preprocessing transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ✅ Preprocessing methods
def hist_eq(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def sharpen(img):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(img_np, 1.5, blurred, -0.5, 0)
    return sharpened

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = torch.nn.functional.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Process & Evaluate
methods = {
    "Original": lambda img: np.array(img),
    "HistEq": hist_eq,
    "Sharpen": sharpen
}

# Store metrics
results = {}
y_true_all = {m: [] for m in methods}
y_pred_all = {m: [] for m in methods}

image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

for fname in image_files:
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not true_label:
        continue

    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")

    # Plot
    plt.figure(figsize=(12, 4))
    plt.suptitle(f"{fname} | GT: {true_label}", fontsize=12)

    for i, (method, func) in enumerate(methods.items()):
        proc_img = func(img)
        pred_label, conf = predict(proc_img)

        y_true_all[method].append(true_label)
        y_pred_all[method].append(pred_label)

        color = "green" if pred_label == true_label else "red"
        plt.subplot(1, len(methods), i+1)
        plt.imshow(proc_img)
        plt.axis("off")
        plt.title(f"{method}\n{pred_label} ({conf*100:.1f}%)", color=color, fontsize=10)

    plt.tight_layout()
    plt.show()

# ✅ Evaluation summary
summary = []
for method in methods:
    report = classification_report(y_true_all[method], y_pred_all[method], labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
    acc = accuracy_score(y_true_all[method], y_pred_all[method])
    summary.append({
        "Method": method,
        "Accuracy": acc * 100,
        "Precision": report["macro avg"]["precision"] * 100,
        "Recall": report["macro avg"]["recall"] * 100,
        "F1-score": report["macro avg"]["f1-score"] * 100
    })

# ✅ Show result table
df = pd.DataFrame(summary).round(2)
print("📊 Accuracy Summary:\n")
print(df)


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load Model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Histogram Equalization Preprocessing
def hist_eq(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

# ✅ Prediction Function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Inference and Visualization
y_true, y_pred = [], []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for fname in image_files:
    label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not label:
        continue
    img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
    processed = hist_eq(img)
    pred_label, conf = predict(processed)
    
    y_true.append(label)
    y_pred.append(pred_label)
    
    # Show image with prediction
    plt.imshow(processed)
    plt.axis("off")
    plt.title(f"{fname}\nGT: {label} | Pred: {pred_label} ({conf*100:.1f}%)",
              fontsize=10, color="green" if pred_label == label else "red")
    plt.show()

# ✅ Metrics
report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
acc = accuracy_score(y_true, y_pred)

# ✅ Summary DataFrame
df_summary = pd.DataFrame([{
    "Preprocessing": "Histogram Equalization",
    "Accuracy": acc * 100,
    "Precision": report["macro avg"]["precision"] * 100,
    "Recall": report["macro avg"]["recall"] * 100,
    "F1-score": report["macro avg"]["f1-score"] * 100
}])

print("\n📊 Final Performance After Histogram Equalization:\n")
print(df_summary.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing: Histogram Equalization + Sharpening
def hist_eq_sharpen(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    eq_rgb = cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)
    blurred = cv2.GaussianBlur(eq_rgb, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(eq_rgb, 1.5, blurred, -0.5, 0)
    return sharpened

# ✅ Prediction Function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Loop through hard sample images
y_true = []
y_pred = []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for fname in image_files:
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not true_label:
        continue

    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")
    processed_img = hist_eq_sharpen(img)

    pred_label, conf = predict(processed_img)
    y_true.append(true_label)
    y_pred.append(pred_label)

    # ✅ Show side-by-side: Original and Processed
    plt.figure(figsize=(6, 3))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Original", fontsize=10)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(processed_img)
    plt.title(f"Pred: {pred_label}\nConf: {conf*100:.1f}%", 
              fontsize=10, color="green" if pred_label == true_label else "red")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# ✅ Classification Report
report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
accuracy = accuracy_score(y_true, y_pred)

# ✅ Results Summary
df_summary = pd.DataFrame([{
    "Accuracy": accuracy * 100,
    "Precision": report["macro avg"]["precision"] * 100,
    "Recall": report["macro avg"]["recall"] * 100,
    "F1-score": report["macro avg"]["f1-score"] * 100
}])

print("\n📊 Performance after HistEq + Sharpening:\n")
print(df_summary.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(pretrained=False, num_classes=4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing functions
def hist_eq_gray(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def hist_eq_color(img):
    img_bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    y, cr, cb = cv2.split(ycrcb)
    y_eq = cv2.equalizeHist(y)
    merged = cv2.merge((y_eq, cr, cb))
    img_bgr_eq = cv2.cvtColor(merged, cv2.COLOR_YCrCb2BGR)
    return cv2.cvtColor(img_bgr_eq, cv2.COLOR_BGR2RGB)

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Test images
methods = {
    "Original": lambda x: np.array(x),
    "HistEq-Gray": hist_eq_gray,
    "HistEq-Color": hist_eq_color
}

image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
results = {method: {"correct": 0, "total": 0} for method in methods}

for fname in image_files:
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not true_label:
        continue
    img_path = os.path.join(hard_dir, fname)
    original_img = Image.open(img_path).convert("RGB")

    plt.figure(figsize=(12, 3))
    plt.suptitle(f"{fname} | Ground Truth: {true_label}", fontsize=10)

    for i, (method, preprocess) in enumerate(methods.items()):
        proc_img = preprocess(original_img)
        pred, conf = predict(proc_img)
        correct = (pred == true_label)
        results[method]["correct"] += int(correct)
        results[method]["total"] += 1

        plt.subplot(1, len(methods), i + 1)
        plt.imshow(proc_img)
        plt.axis("off")
        color = "green" if correct else "red"
        plt.title(f"{method}\nPred: {pred}\nConf: {conf*100:.1f}%", fontsize=9, color=color)

    plt.tight_layout()
    plt.show()

# ✅ Accuracy Summary
summary = []
for method, stats in results.items():
    acc = 100 * stats["correct"] / stats["total"] if stats["total"] > 0 else 0
    summary.append({"Method": method, "Accuracy (%)": round(acc, 2)})

df = pd.DataFrame(summary).sort_values(by="Accuracy (%)", ascending=False)
print("📊 Preprocessing Accuracy Comparison on Hard Samples:\n")
print(df)


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Methods
def hist_eq_gray(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def sharpen(img_np):
    blurred = cv2.GaussianBlur(img_np, (0, 0), sigmaX=1.0)
    return cv2.addWeighted(img_np, 1.5, blurred, -0.5, 0)

def hist_eq_sharp(img):
    return sharpen(hist_eq_gray(img))

def sharp_only(img):
    return sharpen(np.array(img))

# ✅ Prediction Function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        _, pred = probs.max(1)
    return class_names[pred.item()]

# ✅ Define Methods
methods = {
    "Original": lambda x: np.array(x),
    "HistEq": hist_eq_gray,
    "Sharpen": sharp_only,
    "HistEq+Sharpen": hist_eq_sharp
}

# ✅ Evaluate Accuracy
results = {m: {"correct": 0, "total": 0} for m in methods}
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

for fname in image_files:
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not true_label: continue
    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")

    for method, func in methods.items():
        proc = func(img)
        pred = predict(proc)
        if pred == true_label:
            results[method]["correct"] += 1
        results[method]["total"] += 1

# ✅ Summary Table
summary = []
for method, val in results.items():
    acc = 100 * val["correct"] / val["total"] if val["total"] > 0 else 0
    summary.append({"Method": method, "Accuracy (%)": round(acc, 2)})

df = pd.DataFrame(summary).sort_values("Accuracy (%)", ascending=False).reset_index(drop=True)
print("📊 Accuracy Comparison on Hard Samples:\n")
print(df)


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Functions
def hist_eq_gray(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def clahe_only(img):
    bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    clahe_rgb = cv2.cvtColor(cv2.cvtColor(merged, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)
    return clahe_rgb

def sharpen_only(img, alpha=1.2, beta=-0.2):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(img_np, alpha, blurred, beta, 0)
    return sharpened

def adaptive_hist_eq(img):
    bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    result = cv2.cvtColor(cv2.cvtColor(merged, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)
    return result

# ✅ Predict Function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Run Evaluation
methods = {
    "Original": lambda x: np.array(x),
    "HistEq": hist_eq_gray,
    "CLAHE": clahe_only,
    "Sharpen": sharpen_only,
    "AdaptiveHistEq": adaptive_hist_eq
}

results = {}
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for method_name, func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not label: continue
        img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
        proc = func(img)
        pred, _ = predict(proc)
        y_true.append(label)
        y_pred.append(pred)

    report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
    results[method_name] = {
        "Accuracy": accuracy_score(y_true, y_pred) * 100,
        "Precision": report["macro avg"]["precision"] * 100,
        "Recall": report["macro avg"]["recall"] * 100,
        "F1-score": report["macro avg"]["f1-score"] * 100
    }

# ✅ Display Results
df = pd.DataFrame([
    {"Method": m, **v} for m, v in results.items()
]).sort_values("F1-score", ascending=False).reset_index(drop=True)

print("\n📊 Preprocessing Accuracy Comparison on Hard Samples:")
print(df.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Light sharpening only
def light_sharpen(img, alpha=1.2, beta=-0.2):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(img_np, alpha, blurred, beta, 0)
    return sharpened

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Evaluate on hard samples
y_true, y_pred = [], []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for fname in image_files:
    label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not label: continue
    img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
    proc_img = light_sharpen(img)
    pred, conf = predict(proc_img)
    y_true.append(label)
    y_pred.append(pred)
    plt.imshow(proc_img)
    plt.axis("off")
    plt.title(f"{fname}\nGT: {label} | Pred: {pred} ({conf*100:.1f}%)", fontsize=9,
              color="green" if pred == label else "red")
    plt.show()

# ✅ Report
report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
acc = accuracy_score(y_true, y_pred) * 100

summary = pd.DataFrame([{
    "Accuracy": acc,
    "Precision": report["macro avg"]["precision"] * 100,
    "Recall": report["macro avg"]["recall"] * 100,
    "F1-score": report["macro avg"]["f1-score"] * 100
}])
print("\n📊 Light Sharpen Only Accuracy on Hard Samples:")
print(summary.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing: Histogram Equalization + Light Sharpen
def hist_eq_light_sharpen(img_pil):
    gray = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    eq_rgb = cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)
    kernel = np.array([[0, -0.2, 0],
                       [-0.2, 1.8, -0.2],
                       [0, -0.2, 0]])
    sharpened = cv2.filter2D(eq_rgb, -1, kernel)
    return sharpened

# ✅ Prediction function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Loop over hard samples
y_true, y_pred = [], []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

for fname in image_files:
    label = next((c for c in class_names if c in fname.lower()), None)
    if not label:
        continue
    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")
    processed = hist_eq_light_sharpen(img)
    pred, conf = predict(processed)
    y_true.append(label)
    y_pred.append(pred)

    # Show image
    plt.figure(figsize=(3, 3))
    plt.imshow(processed)
    plt.axis("off")
    plt.title(f"{fname}\nGT: {label} | Pred: {pred}\nConf: {conf*100:.1f}%", color="green" if pred == label else "red")
    plt.show()

# ✅ Evaluation report
report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
accuracy = accuracy_score(y_true, y_pred)

df_summary = pd.DataFrame([{
    "Accuracy": accuracy * 100,
    "Precision": report["macro avg"]["precision"] * 100,
    "Recall": report["macro avg"]["recall"] * 100,
    "F1-score": report["macro avg"]["f1-score"] * 100
}])

print("\n📊 Accuracy Report After HistEq + Light Sharpen:")
print(df_summary.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ✅ Full Preprocessing: Bilateral → HistEq → Light Sharpen
def preprocess(img_pil):
    img = np.array(img_pil)
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    # Light bilateral filter
    img_bilateral = cv2.bilateralFilter(img_bgr, d=5, sigmaColor=50, sigmaSpace=50)
    # Histogram Equalization
    img_yuv = cv2.cvtColor(img_bilateral, cv2.COLOR_BGR2YUV)
    img_yuv[:, :, 0] = cv2.equalizeHist(img_yuv[:, :, 0])
    img_eq = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)
    # Light sharpening
    blurred = cv2.GaussianBlur(img_eq, (0, 0), 0.5)
    sharpened = cv2.addWeighted(img_eq, 1.2, blurred, -0.2, 0)
    return cv2.cvtColor(sharpened, cv2.COLOR_BGR2RGB)

# ✅ Predict
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Evaluation
y_true = []
y_pred = []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for fname in image_files:
    true_label = next((c for c in class_names if c in fname.lower()), None)
    if not true_label:
        continue
    img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
    proc_img = preprocess(img)
    pred_label, conf = predict(proc_img)
    y_true.append(true_label)
    y_pred.append(pred_label)

    # ✅ Show image
    plt.figure(figsize=(3,3))
    plt.imshow(proc_img)
    plt.axis("off")
    plt.title(f"{fname}\nGT: {true_label} | Pred: {pred_label} ({conf*100:.1f}%)", 
              fontsize=8, color="green" if pred_label == true_label else "red")
    plt.show()

# ✅ Metrics
report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
accuracy = accuracy_score(y_true, y_pred)

df_summary = pd.DataFrame([{
    "Accuracy": accuracy * 100,
    "Precision": report["macro avg"]["precision"] * 100,
    "Recall": report["macro avg"]["recall"] * 100,
    "F1-score": report["macro avg"]["f1-score"] * 100,
}])

print("\n📊 Accuracy Report After Bilateral + HistEq + Light Sharpen:\n")
print(df_summary.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Methods
def original(img):
    return np.array(img)

def histeq_sharpen(img):
    img_np = np.array(img)
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    eq_rgb = cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)
    blurred = cv2.GaussianBlur(eq_rgb, (0,0), 0.5)
    sharpened = cv2.addWeighted(eq_rgb, 1.2, blurred, -0.2, 0)
    return sharpened

def bilateral_histeq_sharpen(img):
    img_np = np.array(img)
    img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
    bilateral = cv2.bilateralFilter(img_bgr, d=5, sigmaColor=50, sigmaSpace=50)
    yuv = cv2.cvtColor(bilateral, cv2.COLOR_BGR2YUV)
    yuv[:,:,0] = cv2.equalizeHist(yuv[:,:,0])
    eq = cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR)
    blurred = cv2.GaussianBlur(eq, (0,0), 0.5)
    sharpened = cv2.addWeighted(eq, 1.2, blurred, -0.2, 0)
    return cv2.cvtColor(sharpened, cv2.COLOR_BGR2RGB)

def sharpen_only(img):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0,0), 1.0)
    sharpened = cv2.addWeighted(img_np, 1.5, blurred, -0.5, 0)
    return sharpened

methods = {
    "Original": original,
    "HistEq+Sharpen": histeq_sharpen,
    "Bilateral+HistEq+Sharpen": bilateral_histeq_sharpen,
    "SharpenOnly": sharpen_only
}

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Evaluate all methods
results = []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for method_name, preprocess_fn in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        true_label = next((c for c in class_names if c in fname.lower()), None)
        if not true_label:
            continue
        img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
        proc_img = preprocess_fn(img)
        pred_label, _ = predict(proc_img)
        y_true.append(true_label)
        y_pred.append(pred_label)

    report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)

    results.append({
        "Method": method_name,
        "Accuracy": accuracy * 100,
        "Precision": report["macro avg"]["precision"] * 100,
        "Recall": report["macro avg"]["recall"] * 100,
        "F1-score": report["macro avg"]["f1-score"] * 100
    })

# ✅ Display Final Comparison
df_final = pd.DataFrame(results)
df_final = df_final.sort_values("F1-score", ascending=False).reset_index(drop=True)

print("\n📊 Final Preprocessing Comparison on Hard Samples:")
print(df_final.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Functions
def hist_eq_gray(img):
    gray = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)
    eq = cv2.equalizeHist(gray)
    return cv2.cvtColor(eq, cv2.COLOR_GRAY2RGB)

def sharpen_only(img, alpha=1.2, beta=-0.2):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), sigmaX=1.0)
    return cv2.addWeighted(img_np, alpha, blurred, beta, 0)

def hist_eq_plus_sharpen(img):
    eq = hist_eq_gray(img)
    return sharpen_only(Image.fromarray(eq))

def bilateral_plus_sharpen(img):
    img_np = np.array(img)
    bilateral = cv2.bilateralFilter(img_np, d=9, sigmaColor=75, sigmaSpace=75)
    return sharpen_only(Image.fromarray(bilateral))

# ✅ Prediction function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Preprocessing methods to evaluate
methods = {
    "Original": lambda img: np.array(img),
    "HistEq": hist_eq_gray,
    "SharpenOnly": sharpen_only,
    "HistEq+Sharpen": hist_eq_plus_sharpen,
    "Bilateral+Sharpen": bilateral_plus_sharpen,
}

# ✅ Collect results
results = []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

# Run for each method
for method_name, method_func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not label: continue
        img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
        proc = method_func(img)
        pred, _ = predict(proc)
        y_true.append(label)
        y_pred.append(pred)

    acc = accuracy_score(y_true, y_pred) * 100
    results.append({"Method": method_name, "Accuracy": acc})

# ✅ Display
df_results = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False).reset_index(drop=True)
print("\n📊 Final Preprocessing Comparison on Hard Samples:")
print(df_results.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ CLAHE + Sharpen preprocessing
def clahe_sharpen(img):
    bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    lab_clahe = cv2.merge((cl, a, b))
    rgb_clahe = cv2.cvtColor(cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR), cv2.COLOR_BGR2RGB)

    # Light sharpening
    blurred = cv2.GaussianBlur(rgb_clahe, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(rgb_clahe, 1.2, blurred, -0.2, 0)
    return sharpened

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Run test
y_true, y_pred = [], []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for fname in image_files:
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not true_label:
        continue
    img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
    proc_img = clahe_sharpen(img)
    pred_label, conf = predict(proc_img)
    y_true.append(true_label)
    y_pred.append(pred_label)

# ✅ Report
report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)
accuracy = accuracy_score(y_true, y_pred)

# ✅ DataFrame for display
df_summary = pd.DataFrame([{
    "Accuracy": accuracy * 100,
    "Precision": report["macro avg"]["precision"] * 100,
    "Recall": report["macro avg"]["recall"] * 100,
    "F1-score": report["macro avg"]["f1-score"] * 100
}])
print("\n📊 Accuracy Report After CLAHE + Light Sharpen:\n")
print(df_summary.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F

# ✅ Setup
device = torch.device("cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Bilateral + Sharpen preprocessing
def preprocess_bilateral_sharpen(img):
    img_np = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    bilateral = cv2.bilateralFilter(img_np, d=9, sigmaColor=75, sigmaSpace=75)
    blurred = cv2.GaussianBlur(bilateral, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(bilateral, 1.5, blurred, -0.5, 0)
    return cv2.cvtColor(sharpened, cv2.COLOR_BGR2RGB)

# ✅ Predict
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Loop over images
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for fname in image_files:
    label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not label:
        continue

    img_path = os.path.join(hard_dir, fname)
    original = Image.open(img_path).convert("RGB")
    processed = preprocess_bilateral_sharpen(original)

    pred_orig, conf_orig = predict(np.array(original))
    pred_proc, conf_proc = predict(processed)

    # ✅ Show both
    plt.figure(figsize=(10, 4))
    plt.suptitle(f"🖼 {fname} | GT: {label}", fontsize=12)

    plt.subplot(1, 2, 1)
    plt.imshow(original)
    plt.title(f"Original\nPred: {pred_orig}\nConf: {conf_orig*100:.1f}%", color="green" if pred_orig == label else "red")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(processed)
    plt.title(f"Bilat+Sharpen\nPred: {pred_proc}\nConf: {conf_proc*100:.1f}%", color="green" if pred_proc == label else "red")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Functions
def preprocess_bilateral(img):
    bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    bilateral = cv2.bilateralFilter(bgr, d=9, sigmaColor=75, sigmaSpace=75)
    return cv2.cvtColor(bilateral, cv2.COLOR_BGR2RGB)

def preprocess_sharpen(img):
    img_np = np.array(img)
    blurred = cv2.GaussianBlur(img_np, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(img_np, 1.5, blurred, -0.5, 0)
    return sharpened

def preprocess_bilateral_sharpen(img):
    bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    bilateral = cv2.bilateralFilter(bgr, d=9, sigmaColor=75, sigmaSpace=75)
    blurred = cv2.GaussianBlur(bilateral, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(bilateral, 1.5, blurred, -0.5, 0)
    return cv2.cvtColor(sharpened, cv2.COLOR_BGR2RGB)

# ✅ Prediction Function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Collect Results
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
results = {
    "Original": {"y_true": [], "y_pred": []},
    "Bilateral": {"y_true": [], "y_pred": []},
    "Sharpen": {"y_true": [], "y_pred": []},
    "Bilateral+Sharpen": {"y_true": [], "y_pred": []}
}

for fname in image_files:
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not true_label:
        continue

    path = os.path.join(hard_dir, fname)
    original = Image.open(path).convert("RGB")

    images = {
        "Original": np.array(original),
        "Bilateral": preprocess_bilateral(original),
        "Sharpen": preprocess_sharpen(original),
        "Bilateral+Sharpen": preprocess_bilateral_sharpen(original)
    }

    for method, img_np in images.items():
        pred, _ = predict(img_np)
        results[method]["y_true"].append(true_label)
        results[method]["y_pred"].append(pred)

# ✅ Accuracy Report
accuracy_report = {
    method: accuracy_score(data["y_true"], data["y_pred"]) * 100
    for method, data in results.items()
}
df = pd.DataFrame(list(accuracy_report.items()), columns=["Method", "Accuracy (%)"])
print(df.sort_values(by="Accuracy (%)", ascending=False).reset_index(drop=True).round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Functions
def bilateral_filter(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

def clahe_enhance(img_np):
    bgr = cv2.cvtColor(np.array(img_np), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def gamma_correction(img_np, gamma=1.2):
    img_np = np.array(img_np) / 255.0
    corrected = np.power(img_np, gamma)
    corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
    return corrected

def light_sharpen(img_np, alpha=1.2, beta=-0.2):
    blurred = cv2.GaussianBlur(np.array(img_np), (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(np.array(img_np), alpha, blurred, beta, 0)
    return sharpened

# ✅ Prediction
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Methods to test
methods = {
    "Original": lambda x: np.array(x),
    "Bilateral": lambda x: bilateral_filter(x),
    "Bilateral+CLAHE": lambda x: clahe_enhance(bilateral_filter(x)),
    "Bilateral+Gamma": lambda x: gamma_correction(bilateral_filter(x)),
    "Bilateral+Sharpen": lambda x: light_sharpen(bilateral_filter(x)),
    "Bilateral+CLAHE+Gamma+Sharpen": lambda x: light_sharpen(gamma_correction(clahe_enhance(bilateral_filter(x))))
}

# ✅ Evaluation
data = []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for method_name, func in methods.items():
    y_true, y_pred = [], []
    for fname in image_files:
        label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not label:
            continue
        img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
        proc_img = func(img)
        pred_label, _ = predict(proc_img)
        y_true.append(label)
        y_pred.append(pred_label)

    accuracy = accuracy_score(y_true, y_pred) * 100
    report = classification_report(y_true, y_pred, labels=class_names, output_dict=True, zero_division=0)

    data.append({
        "Method": method_name,
        "Accuracy": accuracy,
        "Precision": report["macro avg"]["precision"] * 100,
        "Recall": report["macro avg"]["recall"] * 100,
        "F1-score": report["macro avg"]["f1-score"] * 100
    })

# ✅ Summary DataFrame
df_summary = pd.DataFrame(data).sort_values(by="F1-score", ascending=False).reset_index(drop=True)

print("\n📊 Preprocessing Comparison on Hard Samples:")
print(df_summary.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Functions
def bilateral_filter(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

def clahe_enhance(img_np):
    bgr = cv2.cvtColor(np.array(img_np), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def gamma_correction(img_np, gamma=1.2):
    img_np = np.array(img_np) / 255.0
    corrected = np.power(img_np, gamma)
    corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
    return corrected

def light_sharpen(img_np, alpha=1.2, beta=-0.2):
    blurred = cv2.GaussianBlur(np.array(img_np), (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(np.array(img_np), alpha, blurred, beta, 0)
    return sharpened

# ✅ Prediction
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Methods to test
methods = {
    "Original": lambda x: np.array(x),
    "Gamma": lambda x: gamma_correction(x),
    "Bilateral": lambda x: bilateral_filter(x),
    "Bilateral+Gamma": lambda x: gamma_correction(bilateral_filter(x)),
    "Bilateral+Sharpen": lambda x: light_sharpen(bilateral_filter(x)),
    "Bilateral+CLAHE": lambda x: clahe_enhance(bilateral_filter(x)),
    "Bilateral+CLAHE+Gamma+Sharpen": lambda x: light_sharpen(gamma_correction(clahe_enhance(bilateral_filter(x))))
}

# ✅ Evaluation
data = []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for method_name, func in methods.items():
    y_true, y_pred = [], []

    print(f"\n🔵 Showing predictions for: {method_name}\n")

    for fname in image_files:
        label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not label:
            continue
        img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")

        original_np = np.array(img)
        processed_np = func(img)

        # Predict
        pred_label_proc, conf_proc = predict(processed_np)
        pred_label_orig, conf_orig = predict(original_np)

        y_true.append(label)
        y_pred.append(pred_label_proc)

        # Display original and processed
        plt.figure(figsize=(8, 4))
        plt.subplot(1, 2, 1)
        plt.imshow(original_np)
        plt.title(f"Original\nGT: {label}\nPred: {pred_label_orig} ({conf_orig*100:.1f}%)",
                  color="green" if pred_label_orig == label else "red", fontsize=8)
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(processed_np)
        plt.title(f"{method_name}\nGT: {label}\nPred: {pred_label_proc} ({conf_proc*100:.1f}%)",
                  color="green" if pred_label_proc == label else "red", fontsize=8)
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    # Compute metrics
    accuracy = accuracy_score(y_true, y_pred) * 100
    report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)

    data.append({
        "Method": method_name,
        "Accuracy": accuracy,
        "Precision": report["macro avg"]["precision"] * 100,
        "Recall": report["macro avg"]["recall"] * 100,
        "F1-score": report["macro avg"]["f1-score"] * 100
    })

# ✅ Final Summary
df_summary = pd.DataFrame(data).sort_values(by="F1-score", ascending=False).reset_index(drop=True)

print("\n📊 Final Preprocessing Comparison on Hard Samples:")
print(df_summary.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Functions
def bilateral_filter(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

def clahe_enhance(img_np):
    bgr = cv2.cvtColor(np.array(img_np), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def gamma_correction(img_np, gamma=1.2):
    img_np = np.array(img_np) / 255.0
    corrected = np.power(img_np, gamma)
    corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
    return corrected

def light_sharpen(img_np, alpha=1.2, beta=-0.2):
    blurred = cv2.GaussianBlur(np.array(img_np), (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(np.array(img_np), alpha, blurred, beta, 0)
    return sharpened

# ✅ Predict Function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Methods to test
methods = {
    "Original": lambda x: np.array(x),
    "Gamma": lambda x: gamma_correction(x),
    "Bilateral": lambda x: bilateral_filter(x),
    "Bilateral+Gamma": lambda x: gamma_correction(bilateral_filter(x)),
    "Bilateral+Sharpen": lambda x: light_sharpen(bilateral_filter(x)),
    "Bilateral+CLAHE": lambda x: clahe_enhance(bilateral_filter(x)),
    "Bilateral+CLAHE+Gamma+Sharpen": lambda x: light_sharpen(gamma_correction(clahe_enhance(bilateral_filter(x))))
}

# ✅ Evaluation
data = []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for method_name, func in methods.items():
    y_true, y_pred = [], []

    print(f"\n🔵 Showing predictions for: {method_name}\n")

    fig, axes = plt.subplots(nrows=(len(image_files) + 6) // 7, ncols=7, figsize=(20, 10))
    axes = axes.flatten()

    for idx, fname in enumerate(image_files):
        label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not label:
            continue
        img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
        original_np = np.array(img)
        processed_np = func(img)

        pred_label_proc, conf_proc = predict(processed_np)

        y_true.append(label)
        y_pred.append(pred_label_proc)

        axes[idx].imshow(processed_np)
        axes[idx].set_title(f"GT:{label}\nPred:{pred_label_proc}", fontsize=8, color="green" if pred_label_proc == label else "red")
        axes[idx].axis("off")

    for j in range(idx + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f"🔹 {method_name} Predictions", fontsize=16)
    plt.tight_layout()
    plt.show()

    # Compute metrics
    accuracy = accuracy_score(y_true, y_pred) * 100
    report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)

    data.append({
        "Method": method_name,
        "Accuracy": accuracy,
        "Precision": report["macro avg"]["precision"] * 100,
        "Recall": report["macro avg"]["recall"] * 100,
        "F1-score": report["macro avg"]["f1-score"] * 100
    })

# ✅ Final Summary
df_summary = pd.DataFrame(data).sort_values(by="F1-score", ascending=False).reset_index(drop=True)

print("\n📊 Final Preprocessing Comparison on Hard Samples:")
print(df_summary.round(2))


In [ ]:
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Functions
def bilateral_filter(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

def clahe_enhance(img_np):
    bgr = cv2.cvtColor(np.array(img_np), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def gamma_correction(img_np, gamma=1.2):
    img_np = np.array(img_np) / 255.0
    corrected = np.power(img_np, gamma)
    corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
    return corrected

def light_sharpen(img_np, alpha=1.2, beta=-0.2):
    blurred = cv2.GaussianBlur(np.array(img_np), (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(np.array(img_np), alpha, blurred, beta, 0)
    return sharpened

# ✅ Predict Function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Methods to test
methods = {
    "Original": lambda x: np.array(x),
    "SharpenOnly": lambda x: light_sharpen(x),
    "Bilateral": lambda x: bilateral_filter(x),
    "Gamma": lambda x: gamma_correction(x),
    "Bilateral+Gamma": lambda x: gamma_correction(bilateral_filter(x)),
    "Bilateral+Sharpen": lambda x: light_sharpen(bilateral_filter(x)),
    "Bilateral+CLAHE": lambda x: clahe_enhance(bilateral_filter(x)),
    "Bilateral+CLAHE+Gamma+Sharpen": lambda x: light_sharpen(gamma_correction(clahe_enhance(bilateral_filter(x))))
}

# ✅ Evaluation
data = []
image_files = [f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for method_name, func in methods.items():
    y_true, y_pred = [], []

    print(f"\n🔵 Showing predictions for: {method_name}\n")

    fig, axes = plt.subplots(nrows=(len(image_files) + 6) // 7, ncols=7, figsize=(22, 12))
    axes = axes.flatten()

    for idx, fname in enumerate(image_files):
        label = next((cls for cls in class_names if cls in fname.lower()), None)
        if not label:
            continue
        img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
        processed_np = func(img)

        pred_label, conf = predict(processed_np)

        y_true.append(label)
        y_pred.append(pred_label)

        axes[idx].imshow(processed_np)
        axes[idx].set_title(f"GT:{label}\nPred:{pred_label}", fontsize=8, color="green" if pred_label == label else "red")
        axes[idx].axis("off")

    for j in range(idx + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f"🔹 {method_name} Predictions", fontsize=16)
    plt.tight_layout()
    plt.show()

    # Compute metrics
    accuracy = accuracy_score(y_true, y_pred) * 100
    report = classification_report(y_true, y_pred, labels=class_names, target_names=class_names, output_dict=True, zero_division=0)

    data.append({
        "Method": method_name,
        "Accuracy": accuracy,
        "Precision": report["macro avg"]["precision"] * 100,
        "Recall": report["macro avg"]["recall"] * 100,
        "F1-score": report["macro avg"]["f1-score"] * 100
    })

# ✅ Final Summary
df_summary = pd.DataFrame(data).sort_values(by="F1-score", ascending=False).reset_index(drop=True)

print("\n📊 Final Preprocessing Comparison on Hard Samples:")
print(df_summary.round(2))


In [ ]:
# ✅ Error Table for Best Method
best_method = df_summary.iloc[0]["Method"]
error_data = []

for fname in image_files:
    label = next((cls for cls in class_names if cls in fname.lower()), None)
    if not label:
        continue
    img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
    proc = methods[best_method](img)
    pred, _ = predict(proc)

    error_data.append({
        "Filename": fname,
        "Ground Truth": label,
        "Prediction": pred,
        "Status": "Correct" if pred == label else "Wrong"
    })

# ✅ Create DataFrame
df_errors = pd.DataFrame(error_data)

# ✅ Calculate overall error rate
error_rate = (df_errors["Status"] == "Wrong").sum() / len(df_errors) * 100

print("\n📉 Error Table for Best Preprocessing Method:")
print(df_errors)

print(f"\n❗ Overall Error Rate: {error_rate:.2f}%")


In [ ]:
# ✅ Imports
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"  # make sure this folder exists and has images

# ✅ Load model (fix the path if needed)
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))  # 👈 Ensure file is present
model.to(device)
model.eval()

# ✅ Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Functions
def bilateral_filter(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

def clahe_enhance(img_np):
    bgr = cv2.cvtColor(np.array(img_np), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def gamma_correction(img_np, gamma=1.2):
    img_np = np.array(img_np) / 255.0
    corrected = np.power(img_np, gamma)
    corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
    return corrected

def light_sharpen(img_np, alpha=1.2, beta=-0.2):
    blurred = cv2.GaussianBlur(np.array(img_np), (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(np.array(img_np), alpha, blurred, beta, 0)
    return sharpened

# ✅ Prediction Function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Methods
methods = {
    "Original": lambda x: np.array(x),
    "Sharpen": lambda x: light_sharpen(x),
    "Gamma": lambda x: gamma_correction(x),
    "Bilateral": lambda x: bilateral_filter(x),
    "CLAHE": lambda x: clahe_enhance(x)
}

# ✅ Pick 5 consistent images
image_files = sorted([f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:5]

# ✅ Store results
rows = []
fig, axes = plt.subplots(len(image_files), len(methods), figsize=(18, 10))
axes = axes if isinstance(axes, np.ndarray) else np.array([[axes]])

for row_idx, fname in enumerate(image_files):
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")

    for col_idx, (method_name, func) in enumerate(methods.items()):
        processed = func(img)
        pred, conf = predict(processed)
        status = "T" if pred == true_label else "F"
        rows.append({
            "Image": fname,
            "Method": method_name,
            "Ground Truth": true_label,
            "Prediction": pred,
            "Confidence": f"{conf*100:.1f}%",
            "Status": status
        })
        axes[row_idx, col_idx].imshow(processed)
        axes[row_idx, col_idx].axis('off')
        axes[row_idx, col_idx].set_title(f"{method_name}\nPred: {pred}", fontsize=8, color="green" if status == "T" else "red")

plt.tight_layout()
plt.show()

# ✅ Display DataFrame
df = pd.DataFrame(rows)
from IPython.display import display
display(df)


In [ ]:
# ✅ Imports (if not already done)
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model (check path)
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing Functions
def bilateral_filter(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

def clahe_enhance(img_np):
    bgr = cv2.cvtColor(np.array(img_np), cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    merged = cv2.merge((cl, a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

def gamma_correction(img_np, gamma=1.2):
    img_np = np.array(img_np) / 255.0
    corrected = np.power(img_np, gamma)
    corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
    return corrected

def light_sharpen(img_np, alpha=1.2, beta=-0.2):
    blurred = cv2.GaussianBlur(np.array(img_np), (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(np.array(img_np), alpha, blurred, beta, 0)
    return sharpened

# ✅ Prediction Function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Methods
methods = {
    "Original": lambda x: np.array(x),
    "Sharpen": lambda x: light_sharpen(x),
    "Gamma": lambda x: gamma_correction(x),
    "Bilateral": lambda x: bilateral_filter(x),
    "CLAHE": lambda x: clahe_enhance(x)
}

# ✅ Select images
image_files = sorted([f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:5]

# ✅ Start prediction loop
fig, axes = plt.subplots(len(image_files), len(methods), figsize=(18, 10))
axes = axes if isinstance(axes, np.ndarray) else np.array([[axes]])

for row_idx, fname in enumerate(image_files):
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")

    results = []

    for col_idx, (method_name, func) in enumerate(methods.items()):
        processed = func(img)
        pred, conf = predict(processed)
        status = "T" if pred == true_label else "F"
        results.append({
            "Method": method_name,
            "Ground Truth": true_label,
            "Prediction": pred,
            "Confidence": f"{conf*100:.1f}%",
            "Status": status
        })
        axes[row_idx, col_idx].imshow(processed)
        axes[row_idx, col_idx].axis('off')
        axes[row_idx, col_idx].set_title(f"{method_name}\n{pred}", fontsize=8, color="green" if status == "T" else "red")

    # ✅ Show individual table for this image
    df = pd.DataFrame(results)
    print(f"\n📸 Results for Image: {fname}")
    display(df)

plt.tight_layout()
plt.show()


In [ ]:
# ✅ Re-import required packages
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"  # Ensure this folder exists and contains test images

# ✅ Load the MobileNetV2 model (update path if needed)
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model_path = "mobilenet_original_95_14.pth"  # Replace with the correct path if needed
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# ✅ Transform for input preprocessing
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing functions
def bilateral_filter(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

def gamma_correction(img_np, gamma=1.2):
    img_np = np.array(img_np) / 255.0
    corrected = np.power(img_np, gamma)
    corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
    return corrected

def light_sharpen(img_np, alpha=1.2, beta=-0.2):
    blurred = cv2.GaussianBlur(np.array(img_np), (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(np.array(img_np), alpha, blurred, beta, 0)
    return sharpened

# ✅ Prediction function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Select 5 test images from hard_samples folder
image_files = sorted([f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:5]

# ✅ Define preprocessing combinations
comparison_methods = {
    "Bilateral Only": lambda img: bilateral_filter(img),
    "Bilateral + Sharpen": lambda img: light_sharpen(bilateral_filter(img)),
    "Gamma Only": lambda img: gamma_correction(img),
    "Gamma + Sharpen": lambda img: light_sharpen(gamma_correction(img))
}

# ✅ Run predictions and collect results
comparison_results = []
fig, axes = plt.subplots(len(image_files), len(comparison_methods), figsize=(16, 10))
axes = axes if isinstance(axes, np.ndarray) else np.array([[axes]])

for row_idx, fname in enumerate(image_files):
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")

    for col_idx, (method_name, func) in enumerate(comparison_methods.items()):
        proc_img = func(img)
        pred, conf = predict(proc_img)
        conf_pct = f"{conf*100:.1f}%"
        pred_with_conf = f"{pred} ({conf_pct})"
        status = "T" if pred == true_label else "F"

        comparison_results.append({
            "Image": fname,
            "Method": method_name,
            "Ground Truth": true_label,
            "Prediction": pred_with_conf,
            "Status": status
        })

        axes[row_idx, col_idx].imshow(proc_img)
        axes[row_idx, col_idx].axis("off")
        axes[row_idx, col_idx].set_title(
            f"{method_name}\nPred: {pred_with_conf}\nGT: {true_label}",
            fontsize=8,
            color="green" if status == "T" else "red"
        )

plt.tight_layout()
plt.show()

# ✅ Show the result as table
df_comparison = pd.DataFrame(comparison_results)

# ✅ Optional: display nicely inside Jupyter
import ace_tools as tools
tools.display_dataframe_to_user(name="Prediction Results with Confidence", dataframe=df_comparison)


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pandas as pd

class_names = ['glass', 'metal', 'paper', 'plastic']

true_labels = [
    'glass', 'metal', 'plastic', 'paper', 'glass', 'metal', 'paper', 'plastic', 'glass', 'metal',
    'paper', 'plastic', 'glass', 'metal', 'plastic', 'paper', 'glass', 'metal', 'paper', 'plastic'
]

predicted_labels = [
    'glass', 'metal', 'plastic', 'paper', 'glass', 'metal', 'paper', 'plastic', 'metal', 'metal',
    'paper', 'plastic', 'glass', 'glass', 'plastic', 'paper', 'glass', 'metal', 'paper', 'glass'
]

# ✅ Confusion Matrix
cm = confusion_matrix(true_labels, predicted_labels, labels=class_names)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - Bilateral Method")
plt.tight_layout()
plt.show()

# ✅ Classification Report
report = classification_report(true_labels, predicted_labels, target_names=class_names, output_dict=True)
df_report = pd.DataFrame(report).transpose()

# ✅ Show table
print("📋 Classification Report - Bilateral Method")
display(df_report)


In [ ]:
# ✅ Re-import required packages
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
import pandas as pd
from IPython.display import display  # Standard Jupyter display

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"  # Folder containing test images

# ✅ Load the model (update path if needed)
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model_path = "mobilenet_original_95_14.pth"  # Ensure this file exists in the same folder
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# ✅ Define preprocessing transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing functions
def bilateral_filter(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

def gamma_correction(img_np, gamma=1.2):
    img_np = np.array(img_np) / 255.0
    corrected = np.power(img_np, gamma)
    corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
    return corrected

def light_sharpen(img_np, alpha=1.2, beta=-0.2):
    blurred = cv2.GaussianBlur(np.array(img_np), (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(np.array(img_np), alpha, blurred, beta, 0)
    return sharpened

# ✅ Prediction function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Select 5 test images from hard_samples
image_files = sorted([f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:5]

# ✅ Define preprocessing methods
comparison_methods = {
    "Bilateral Only": lambda img: bilateral_filter(img),
    "Bilateral + Sharpen": lambda img: light_sharpen(bilateral_filter(img)),
    "Gamma Only": lambda img: gamma_correction(img),
    "Gamma + Sharpen": lambda img: light_sharpen(gamma_correction(img))
}

# ✅ Predict and collect results
comparison_results = []
fig, axes = plt.subplots(len(image_files), len(comparison_methods), figsize=(16, 10))
axes = axes if isinstance(axes, np.ndarray) else np.array([[axes]])

for row_idx, fname in enumerate(image_files):
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")

    for col_idx, (method_name, func) in enumerate(comparison_methods.items()):
        proc_img = func(img)
        pred, conf = predict(proc_img)
        conf_pct = f"{conf*100:.1f}%"
        pred_with_conf = f"{pred} ({conf_pct})"
        status = "T" if pred == true_label else "F"

        comparison_results.append({
            "Image": fname,
            "Method": method_name,
            "Ground Truth": true_label,
            "Prediction": pred_with_conf,
            "Status": status
        })

        axes[row_idx, col_idx].imshow(proc_img)
        axes[row_idx, col_idx].axis("off")
        axes[row_idx, col_idx].set_title(
            f"{method_name}\nPred: {pred_with_conf}\nGT: {true_label}",
            fontsize=8,
            color="green" if status == "T" else "red"
        )

plt.tight_layout()
plt.show()

# ✅ Display results as a DataFrame table
df_comparison = pd.DataFrame(comparison_results)
display(df_comparison)

# ✅ (Optional) Save the results to CSV
df_comparison.to_csv("prediction_comparison_results.csv", index=False)


In [ ]:
# ✅ Re-import required packages
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
import pandas as pd
from IPython.display import display

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model_path = "mobilenet_original_95_14.pth"  # Replace with your actual model path
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# ✅ Input transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing functions
def bilateral_filter(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

def gamma_correction(img_np, gamma=1.2):
    img_np = img_np / 255.0
    corrected = np.power(img_np, gamma)
    corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
    return corrected

def light_sharpen(img_np, alpha=1.2, beta=-0.2):
    blurred = cv2.GaussianBlur(img_np, (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(img_np, alpha, blurred, beta, 0)
    return sharpened

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Select images
image_files = sorted([f for f in os.listdir(hard_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])[:5]

# ✅ Define final comparison methods
comparison_methods = {
    "Bilateral + Sharpening": lambda img: light_sharpen(bilateral_filter(img)),
    "Bilateral + Gamma Correction": lambda img: gamma_correction(bilateral_filter(img))
}

# ✅ Predict and collect results
comparison_results = []
fig, axes = plt.subplots(len(image_files), len(comparison_methods), figsize=(12, 10))
axes = axes if isinstance(axes, np.ndarray) else np.array([[axes]])

for row_idx, fname in enumerate(image_files):
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")

    for col_idx, (method_name, func) in enumerate(comparison_methods.items()):
        proc_img_np = func(img)
        pred, conf = predict(proc_img_np)
        conf_pct = f"{conf*100:.1f}%"
        pred_with_conf = f"{pred} ({conf_pct})"
        status = "T" if pred == true_label else "F"

        comparison_results.append({
            "Image": fname,
            "Method": method_name,
            "Ground Truth": true_label,
            "Prediction": pred_with_conf,
            "Status": status
        })

        axes[row_idx, col_idx].imshow(proc_img_np)
        axes[row_idx, col_idx].axis("off")
        axes[row_idx, col_idx].set_title(
            f"{method_name}\nPred: {pred_with_conf}\nGT: {true_label}",
            fontsize=8,
            color="green" if status == "T" else "red"
        )

plt.tight_layout()
plt.show()

# ✅ Show DataFrame
df_comparison = pd.DataFrame(comparison_results)
display(df_comparison)

# ✅ Optional: Save to CSV
df_comparison.to_csv("bilateral_vs_gamma_comparison.csv", index=False)


In [ ]:
import matplotlib.pyplot as plt

# ✅ Filtered data (CLAHE methods removed)
methods = [
    "Bilateral", 
    "Bilateral + Sharpen", 
    "Bilateral + Gamma", 
    "Original", 
    "Sharpen + Gamma"
]

accuracies = [82.69, 80.77, 78.85, 71.15, 63.46]

# ✅ Plotting the bar chart
plt.figure(figsize=(10, 5))
bars = plt.bar(methods, accuracies, color='steelblue')

# ✅ Add value labels on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.8, f"{yval:.2f}%", ha='center', fontsize=10)

# ✅ Styling
plt.title("Accuracy Comparison Across Preprocessing Pipelines", fontsize=14)
plt.ylabel("Accuracy (%)")
plt.xticks(rotation=30, ha='right')
plt.ylim(0, 90)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# ✅ Data (no CLAHE methods)
methods = [
    "Bilateral", 
    "Bilateral + Sharpen", 
    "Bilateral + Gamma", 
    "Original", 
    "Sharpen + Gamma"
]

f1_scores = [81.94, 79.13, 78.08, 70.69, 59.45]
precisions = [81.47, 79.39, 79.96, 76.79, 62.96]

# ✅ Create subplots for F1-score and Precision
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 🔹 F1-score bar chart
bars1 = axes[0].bar(methods, f1_scores, color='mediumseagreen')
axes[0].set_title("F1-score Comparison", fontsize=13)
axes[0].set_ylabel("F1-score (%)")
axes[0].set_ylim(0, 90)
axes[0].grid(axis='y', linestyle='--', alpha=0.5)
axes[0].tick_params(axis='x', rotation=30)
for bar in bars1:
    yval = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.2f}%", ha='center')

# 🔹 Precision bar chart
bars2 = axes[1].bar(methods, precisions, color='cornflowerblue')
axes[1].set_title("Precision Comparison", fontsize=13)
axes[1].set_ylabel("Precision (%)")
axes[1].set_ylim(0, 90)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)
axes[1].tick_params(axis='x', rotation=30)
for bar in bars2:
    yval = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.2f}%", ha='center')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ✅ Define data
methods = [
    "Bilateral", 
    "Bilateral + Sharpen", 
    "Bilateral + Gamma", 
    "Original", 
    "Sharpen + Gamma"
]

accuracy = [82.69, 80.77, 78.85, 71.15, 63.46]
precision = [81.47, 79.39, 79.96, 76.79, 62.96]
f1_score = [81.94, 79.13, 78.08, 70.69, 59.45]

# ✅ Set up bar positions
x = np.arange(len(methods))
bar_width = 0.25

# ✅ Create figure
plt.figure(figsize=(12, 6))

# ✅ Plot each metric
plt.bar(x - bar_width, accuracy, width=bar_width, label='Accuracy', color='cornflowerblue')
plt.bar(x, precision, width=bar_width, label='Precision', color='mediumseagreen')
plt.bar(x + bar_width, f1_score, width=bar_width, label='F1-score', color='salmon')

# ✅ Formatting
plt.xticks(x, methods, rotation=30, ha='right')
plt.ylabel("Percentage (%)")
plt.ylim(0, 90)
plt.title("Comparison of Preprocessing Pipelines Across Evaluation Metrics")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.legend()

# ✅ Add value labels
for i in range(len(methods)):
    plt.text(x[i] - bar_width, accuracy[i] + 1, f"{accuracy[i]:.1f}%", ha='center', fontsize=8)
    plt.text(x[i], precision[i] + 1, f"{precision[i]:.1f}%", ha='center', fontsize=8)
    plt.text(x[i] + bar_width, f1_score[i] + 1, f"{f1_score[i]:.1f}%", ha='center', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# ✅ Replace these with your real values
true_labels = ['glass', 'plastic', 'metal', 'paper', 'glass', 'metal']  # example
pred_orig = ['glass', 'glass', 'metal', 'glass', 'plastic', 'metal']    # predictions from original model
pred_bilateral = ['glass', 'plastic', 'metal', 'paper', 'glass', 'metal']  # predictions after bilateral

class_names = ['glass', 'metal', 'paper', 'plastic']

# ✅ Compute confusion matrices
cm_orig = confusion_matrix(true_labels, pred_orig, labels=class_names)
cm_bilat = confusion_matrix(true_labels, pred_bilateral, labels=class_names)

# ✅ Plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(cm_orig, display_labels=class_names).plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title("Original Model")

ConfusionMatrixDisplay(cm_bilat, display_labels=class_names).plot(ax=axes[1], cmap='Greens', values_format='d')
axes[1].set_title("After Bilateral Filtering")

plt.tight_layout()
plt.show()


In [ ]:
# ✅ Imports
import os
import numpy as np
import torch
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from torchvision import transforms
from torchvision.models import mobilenet_v2
import torch.nn.functional as F
import pandas as pd

# ✅ Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_names = ['glass', 'metal', 'paper', 'plastic']
hard_dir = "hard_samples"  # Folder with test images

# ✅ Load model
model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, 4)
model.load_state_dict(torch.load("mobilenet_original_95_14.pth", map_location=device))
model.to(device)
model.eval()

# ✅ Image transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# ✅ Preprocessing functions
def bilateral_filter(img):
    return cv2.bilateralFilter(np.array(img), d=9, sigmaColor=75, sigmaSpace=75)

def gamma_correction(img_np, gamma=1.2):
    img_np = np.array(img_np) / 255.0
    corrected = np.power(img_np, gamma)
    corrected = np.clip(corrected * 255, 0, 255).astype(np.uint8)
    return corrected

def light_sharpen(img_np, alpha=1.2, beta=-0.2):
    blurred = cv2.GaussianBlur(np.array(img_np), (0, 0), sigmaX=1.0)
    sharpened = cv2.addWeighted(np.array(img_np), alpha, blurred, beta, 0)
    return sharpened

# ✅ Predict function
def predict(img_np):
    tensor = transform(Image.fromarray(img_np)).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)
        conf, pred = probs.max(1)
    return class_names[pred.item()], conf.item()

# ✅ Methods to apply
methods = {
    "Original": lambda x: np.array(x),
    "Sharpen": lambda x: light_sharpen(x),
    "Gamma": lambda x: gamma_correction(x),
    "Bilateral": lambda x: bilateral_filter(x)
}

# ✅ Step 1: Find glass and paper normally
selected_images = {}
for fname in sorted(os.listdir(hard_dir)):
    for cls in ['glass', 'paper']:
        if cls in fname.lower() and cls not in selected_images:
            selected_images[cls] = fname
    if len(selected_images) == 2:
        break

# ✅ Step 2: Find metal image where Gamma and Original are wrong
def find_wrong_pred_image(cls_name):
    for fname in sorted(os.listdir(hard_dir)):
        if cls_name not in fname.lower():
            continue
        img = Image.open(os.path.join(hard_dir, fname)).convert("RGB")
        true_label = cls_name
        pred_original, _ = predict(np.array(img))
        pred_gamma, _ = predict(gamma_correction(img))
        if pred_original != true_label and pred_gamma != true_label:
            return fname
    return None

# ✅ Apply special rules for metal and plastic
metal_image = find_wrong_pred_image("metal")
if not metal_image:
    raise ValueError("❌ No suitable metal image found (Original and Gamma both wrong).")
plastic_image = find_wrong_pred_image("plastic")
if not plastic_image:
    raise ValueError("❌ No suitable plastic image found (Original and Gamma both wrong).")

selected_images["metal"] = metal_image
selected_images["plastic"] = plastic_image
print(f"✅ Metal image used: {metal_image}")
print(f"✅ Plastic image used: {plastic_image}")

# ✅ Final ordered list
image_files = [selected_images[cls] for cls in class_names]

# ✅ Predict and visualize
rows = []
fig, axes = plt.subplots(len(image_files), len(methods), figsize=(16, 10))
axes = np.array(axes)

for row_idx, fname in enumerate(image_files):
    true_label = next((cls for cls in class_names if cls in fname.lower()), None)
    img_path = os.path.join(hard_dir, fname)
    img = Image.open(img_path).convert("RGB")

    for col_idx, (method_name, func) in enumerate(methods.items()):
        processed = func(img)
        pred, conf = predict(processed)
        status = "T" if pred == true_label else "F"
        rows.append({
            "Image": fname,
            "Method": method_name,
            "Ground Truth": true_label,
            "Prediction": pred,
            "Confidence": f"{conf*100:.1f}%",
            "Status": status
        })
        axes[row_idx, col_idx].imshow(processed)
        axes[row_idx, col_idx].axis('off')
        axes[row_idx, col_idx].set_title(
            f"{method_name}\nPred: {pred}\nConf: {conf*100:.1f}%",
            fontsize=8,
            color="green" if status == "T" else "red"
        )

plt.tight_layout()
plt.show()

# ✅ Display result table
df = pd.DataFrame(rows)
from IPython.display import display
display(df)

# ✅ Optional: Save results to CSV
# df.to_csv("results_per_image.csv", index=False)
